# 06 · Personalized Meta-RL Strategy Selector (PPO)

**Project:** Personalized AI Trading Agent · **Notebook 6 of 6 (final)**
**Algorithm:** PPO (Stable-Baselines3, discrete actions) · **Artifact:** `meta_rl_selector.zip`

The agent **does not predict BUY / SELL / HOLD / EXIT**. It selects *which of five frozen, pre-trained specialist policies* controls the next decision:

| ID | Policy | ID | Policy |
|---|---|---|---|
| 0 | Momentum | 3 | Breakout |
| 1 | Mean-Reversion | 4 | Conservative Risk-Aware |
| 2 | Trend-Following | | |

The chosen policy *proposes* an action; a **separate deterministic risk controller** can approve, clip, reject or override it before simulated/paper execution. PPO can never bypass that gate.

### Status and honesty rules
* This is a **research prototype for offline evaluation and paper trading**. It is not a live-trading bot, and nothing here implies live profitability.
* Every performance number is computed at run time from held-out data. Nothing is hard-coded, and the notebook states a "superiority" conclusion only if the held-out numbers support it.
* If any of the five policy artifacts is missing, the notebook **stops with an error**. It never substitutes random or placeholder models.
* Unless you supply a real `trader_history_csv`, investor profiles are **synthetic** and labelled as such. The notebook never claims to have learned a real individual's behaviour from synthetic data.

### Notebook map
1. Installation & configuration → 2. Artifact upload & validation → 3. Data loading & features → 4. Observation construction → 6. Risk controller *(defined before the environment because the environment depends on it)* → 5. Gymnasium environment → 7. PPO training → 8. Validation → 9. Held-out backtest → 10. Baselines & ablations → 11. Visualisation → 12. Export → 13. Inference demo → Integration report.

### Required inputs (the contract from notebooks 1-5)
For each stem in `momentum_policy`, `mean_reversion_policy`, `trend_following_policy`, `breakout_policy`, `conservative_policy`:

| File | Content |
|---|---|
| `<stem>.pt` | TorchScript module, a pickled `nn.Module`, or `{"state_dict":…, "arch":{type: mlp/lstm/gru, input_dim, hidden_dim, num_layers, n_actions, seq_len}}` |
| `<stem>_config.json` | `action_map` (`{"0":"HOLD","1":"BUY","2":"SELL","3":"EXIT"}`), optional `output_type` (`logits`/`probabilities`), `input_mode` (`sequence`/`flat`), `train_end_date`, `preprocessing` |
| `<stem>_feature_schema.json` | `feature_columns` (ordered list), `seq_len` |
| `<stem>_preprocessor.pkl` | fitted scaler exposing `.transform` (joblib/pickle) |

If your earlier notebooks used other file names or layouts, edit `ARTIFACT_FILES` and the loader in section 2. The validation step tells you exactly what is wrong.

In [ ]:
# ---- 1a. Installation (Colab). torch is preinstalled on Colab; CPU is fine for this MLP-PPO workload. ----
!pip -q install "gymnasium>=0.29" "stable-baselines3>=2.3" yfinance scikit-learn joblib matplotlib pandas pyarrow

In [ ]:
# ---- 1b. Imports, constants and configuration ----
import os, sys, json, math, time, random, hashlib, warnings, pickle, copy, platform
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import gymnasium as gym
from gymnasium import spaces
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.logger import configure as sb3_configure
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*valid feature names.*")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

# ---------- Canonical constants (never change after training) ----------
ACTIONS = ["HOLD", "BUY", "SELL", "EXIT"]            # canonical action order used by the environment
HOLD, BUY, SELL, EXIT = range(4)
POLICY_NAMES = ["Momentum Policy", "Mean-Reversion Policy", "Trend-Following Policy",
                "Breakout Policy", "Conservative Risk-Aware Policy"]
SELECT_NAMES = ["Select Momentum", "Select Mean-Reversion", "Select Trend-Following",
                "Select Breakout", "Select Conservative"]
N_POLICIES = len(POLICY_NAMES)
ARTIFACT_FILES = {
    i: {"model": f"{stem}.pt", "config": f"{stem}_config.json",
        "schema": f"{stem}_feature_schema.json", "preprocessor": f"{stem}_preprocessor.pkl"}
    for i, stem in enumerate(["momentum_policy", "mean_reversion_policy", "trend_following_policy",
                              "breakout_policy", "conservative_policy"])
}

# ---------- Configuration objects ----------
@dataclass
class PathsConfig:
    artifact_source: str = "drive"        # "drive" | "upload" | "local"
    drive_artifact_dir: str = "/content/drive/MyDrive/personalized_trading_agent/artifacts"
    local_artifact_dir: str = "/content/artifacts"
    output_dir: str = "/content/meta_rl_outputs"
    drive_output_dir: Optional[str] = "/content/drive/MyDrive/personalized_trading_agent/meta_rl_outputs"  # None = don't copy
    market_data_dir: str = "/content/market_data"      # used for csv source and as a download cache
    policy_features_dir: Optional[str] = None          # optional: <TICKER>_policy_features.(parquet|csv) exported by earlier notebooks
    behavior_encoder_path: Optional[str] = None        # TorchScript encoder from an earlier notebook; None = no embeddings
    trader_history_csv: Optional[str] = None           # real trade history (date,action,size[,high_vol,in_drawdown]); None = synthetic profiles only

@dataclass
class DataConfig:
    source: str = "yfinance"              # "yfinance" | "csv" (csv: <market_data_dir>/<TICKER>.csv with Date,Open,High,Low,Close,Volume)
    tickers: List[str] = field(default_factory=lambda: ["AAPL", "MSFT", "AMZN", "JPM", "XOM", "JNJ", "NVDA", "KO"])
    benchmark: str = "SPY"
    start: str = "2010-01-01"
    end: Optional[str] = None
    train_end: str = "2020-12-31"         # train: <= train_end
    val_end: str = "2022-12-31"           # validation: (train_end + embargo, val_end]; test: (val_end + embargo, end]
    embargo_days: int = 5
    min_split_days: int = 120
    strict_leakage_check: bool = True     # refuse to run if test overlaps the specialist policies' training period

@dataclass
class EnvConfig:
    initial_capital: float = 100_000.0
    fee_bps: float = 5.0                  # per-side commission, basis points of notional
    slippage_bps: float = 5.0             # adverse fill vs next open
    episode_len: int = 252                # training episode length (trading days)
    checkpoint_days: int = 21             # investor review checkpoint cadence
    use_trader_profile: bool = True       # ablation switch: behaviour block
    use_candidate_perf: bool = True       # ablation switch: shadow performance block

@dataclass
class RewardConfig:
    perf_mode: str = "dsr"                # "dsr" (differential Sharpe ratio) | "mean_variance"
    dsr_eta: float = 0.02
    dsr_warmup: int = 20
    dsr_clip: float = 5.0
    mv_lambda: float = 1.0                # mean-variance risk aversion (used if perf_mode == "mean_variance")
    w_perf: float = 1.0                   # risk-adjusted performance
    w_dd: float = 2.0                     # drawdown penalty
    dd_soft_frac: float = 0.25            # drawdown tolerated up to this fraction of the investor's tolerance before the level penalty starts
    w_cost: float = 1.0                   # transaction costs (per 10 bps of equity)
    w_turnover: float = 1.0               # |delta position fraction|
    w_behavior: float = 0.05              # behavioural compatibility (kept small and separately measured)
    w_reject: float = 0.10                # penalty when the risk gate rejects/overrides a proposal

@dataclass
class RiskConfig:
    allowed_modes: Tuple[str, ...] = ("simulation", "paper")   # live mode is intentionally NOT authorised
    mode: str = "simulation"
    trading_enabled: bool = True          # global kill switch
    order_fraction: float = 0.05          # standard order size (fraction of equity)
    max_position_size: float = 0.30       # fraction of equity in one asset
    max_portfolio_exposure: float = 0.30  # fraction of equity invested overall
    max_daily_loss: float = 0.015         # no new BUYs after a daily equity loss >= this
    max_drawdown: float = 0.20            # circuit breaker (min with the investor's own tolerance)
    max_ann_vol_new_buys: float = 0.80    # annualised vol ceiling for new BUYs (scaled by investor risk tolerance)
    min_order_fraction: float = 0.005

@dataclass
class ProfileConfig:
    n_train_per_archetype: int = 8
    n_val_per_archetype: int = 1
    n_test_per_archetype: int = 2
    seeds: Tuple[int, int, int] = (101, 202, 303)

@dataclass
class PPOConfig:
    total_timesteps: int = 400_000
    n_envs: int = 4
    n_steps: int = 512
    batch_size: int = 256
    n_epochs: int = 10
    learning_rate: float = 3e-4
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    net_arch: Tuple[int, ...] = (128, 128)
    device: str = "cpu"                   # "cpu" is usually fastest for small MLPs; "cuda"/"auto" also supported
    eval_freq: int = 25_000               # validation every N total timesteps
    patience: int = 6                     # early stop after this many validations without improvement ...
    min_evals: int = 8                    # ... but only after at least this many validations
    selection_metric: str = "mean_reward" # "mean_reward" | "mean_sharpe"
    sens_timesteps: int = 100_000         # budget for reward-sensitivity variants

@dataclass
class MasterConfig:
    paths: PathsConfig = field(default_factory=PathsConfig)
    data: DataConfig = field(default_factory=DataConfig)
    env: EnvConfig = field(default_factory=EnvConfig)
    reward: RewardConfig = field(default_factory=RewardConfig)
    risk: RiskConfig = field(default_factory=RiskConfig)
    profiles: ProfileConfig = field(default_factory=ProfileConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    seed: int = 42
    quick_run: bool = False               # True = tiny budgets for a fast end-to-end smoke run (results NOT meaningful)
    run_sensitivity: bool = True
    run_ablations: bool = True

CFG = MasterConfig()
if CFG.quick_run:
    CFG.ppo.total_timesteps, CFG.ppo.eval_freq, CFG.ppo.patience, CFG.ppo.min_evals, CFG.ppo.sens_timesteps = 40_000, 8_000, 3, 2, 16_000

def set_global_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    set_random_seed(seed)

set_global_seed(CFG.seed)
OUT = Path(CFG.paths.output_dir); OUT.mkdir(parents=True, exist_ok=True)
print(f"torch {torch.__version__} | sb3 {sb3.__version__} | gymnasium {gym.__version__} | device for inference: cpu | cuda available: {torch.cuda.is_available()}")
print("Outputs ->", OUT)

## 2. Artifact upload and validation

Policies are loaded from Google Drive, an upload dialog, or a local folder. The notebook validates that **all five** share feature columns (same order), sequence length and action mapping, and that each one runs a forward pass of the right shape. A missing file raises `ArtifactError` with upload instructions; there is no fallback to random weights.

> **Security note:** `torch.load(..., weights_only=False)` and `joblib.load` unpickle arbitrary objects. Only load artifacts you produced yourself.

In [ ]:
# ---- 2. Artifact acquisition, validation and loading ----
class ArtifactError(RuntimeError):
    """Raised when a required upstream artifact is missing or incompatible. Never swallowed."""

def sha256_of(path, chunk=1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def _required_paths(d: Path) -> Dict[Tuple[int, str], Path]:
    return {(i, k): d / fn for i, files in ARTIFACT_FILES.items() for k, fn in files.items()}

def resolve_artifact_dir(P: PathsConfig) -> Path:
    if P.artifact_source == "drive":
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except ImportError:
            print("Not running on Colab: treating drive_artifact_dir as an ordinary local path.")
        return Path(P.drive_artifact_dir)
    if P.artifact_source in ("upload", "local"):
        d = Path(P.local_artifact_dir); d.mkdir(parents=True, exist_ok=True); return d
    raise ValueError("artifact_source must be 'drive', 'upload' or 'local'")

def ensure_artifacts(P: PathsConfig) -> Path:
    d = resolve_artifact_dir(P)
    missing = [p for p in _required_paths(d).values() if not p.exists()]
    if missing and P.artifact_source == "upload":
        from google.colab import files  # type: ignore
        print("Missing files:\n  " + "\n  ".join(p.name for p in missing))
        print("Select them in the upload dialog below ...")
        for name, data in files.upload().items():
            (d / Path(name).name).write_bytes(data)
        missing = [p for p in _required_paths(d).values() if not p.exists()]
    if missing:
        raise ArtifactError(
            "Required specialist-policy artifacts are missing from " + str(d) + ":\n  "
            + "\n  ".join(p.name for p in missing)
            + "\n\nHow to fix: (a) copy these files from the output of notebooks 1-5 into that folder; or "
              "(b) set CFG.paths.artifact_source = 'upload' and re-run this cell to upload them; or "
              "(c) point CFG.paths.drive_artifact_dir / local_artifact_dir at the correct folder. "
              "This notebook will NOT substitute random or placeholder models.")
    return d

def _load_json(p: Path) -> dict:
    try:
        return json.loads(Path(p).read_text())
    except Exception as e:
        raise ArtifactError(f"Cannot parse JSON file {p}: {e}")

class SeqPolicyNet(nn.Module):
    """Reference architecture for checkpoints saved as {'arch': {...}, 'state_dict': {...}}.
    mlp -> keys net.*   |   lstm/gru -> keys rnn.* and head.*   (inputs are (N, seq_len, n_features))."""
    def __init__(self, arch: dict):
        super().__init__()
        self.kind = str(arch.get("type", "mlp")).lower()
        d_in, A = int(arch["input_dim"]), int(arch["n_actions"])
        h, L = int(arch.get("hidden_dim", 64)), int(arch.get("num_layers", 2))
        drop, seq = float(arch.get("dropout", 0.0)), int(arch.get("seq_len", 1))
        if self.kind == "mlp":
            layers, d = [], d_in * seq
            for _ in range(L):
                layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(drop)]; d = h
            layers.append(nn.Linear(d, A)); self.net = nn.Sequential(*layers)
        elif self.kind in ("lstm", "gru"):
            cell = nn.LSTM if self.kind == "lstm" else nn.GRU
            self.rnn = cell(d_in, h, num_layers=L, batch_first=True, dropout=drop if L > 1 else 0.0)
            self.head = nn.Linear(h, A)
        else:
            raise ArtifactError(f"Unsupported arch type '{self.kind}'. Export a TorchScript module instead.")
    def forward(self, x):
        if self.kind == "mlp":
            return self.net(x.reshape(x.shape[0], -1))
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])

def load_policy_module(path: Path, cfg: dict, default_arch: dict):
    p = str(path)
    try:
        m = torch.jit.load(p, map_location="cpu"); m.eval(); return m, "torchscript"
    except Exception:
        pass
    obj = torch.load(p, map_location="cpu", weights_only=False)
    if isinstance(obj, nn.Module):
        obj.eval(); return obj, "pickled_nn_module"
    if isinstance(obj, dict) and "state_dict" in obj:
        arch = dict(default_arch); arch.update(obj.get("arch") or cfg.get("arch") or {})
        net = SeqPolicyNet(arch)
        try:
            net.load_state_dict(obj["state_dict"], strict=True)
        except Exception as e:
            raise ArtifactError(f"{Path(p).name}: state_dict does not match the reference SeqPolicyNet({arch}). "
                                f"Either fix 'arch' in the checkpoint or re-export the policy as TorchScript "
                                f"(torch.jit.script/trace) from its training notebook. Detail: {e}")
        net.eval(); return net, "state_dict+SeqPolicyNet"
    raise ArtifactError(f"{Path(p).name}: unsupported checkpoint format ({type(obj).__name__}). "
                        "Expected TorchScript, a pickled nn.Module, or a dict with 'state_dict' and 'arch'.")

class FrozenPolicy:
    """A frozen specialist policy. Produces probabilities in the *canonical* action order (HOLD,BUY,SELL,EXIT)."""
    def __init__(self, idx, stem, module, kind, columns, seq_len, action_map, scaler, input_mode, output_type, files):
        self.idx, self.stem, self.module, self.kind = idx, stem, module, kind
        self.columns, self.seq_len, self.action_map = list(columns), int(seq_len), dict(action_map)
        self.scaler, self.input_mode, self.output_type, self.files = scaler, input_mode, output_type, files
        name_to_model_idx = {v: k for k, v in self.action_map.items()}
        self.perm = {ci: name_to_model_idx[a] for ci, a in enumerate(ACTIONS) if a in name_to_model_idx}
        for prm in self.module.parameters(): prm.requires_grad_(False)   # frozen
        self.module.eval()
    def transform(self, X: np.ndarray) -> np.ndarray:
        return X.astype(np.float32) if self.scaler is None else np.asarray(self.scaler.transform(X), dtype=np.float32)
    @torch.no_grad()
    def predict_proba(self, windows: np.ndarray, batch: int = 2048) -> np.ndarray:
        outs = []
        for i in range(0, len(windows), batch):
            x = torch.as_tensor(np.ascontiguousarray(windows[i:i + batch]), dtype=torch.float32)
            if self.input_mode == "flat": x = x.reshape(x.shape[0], -1)
            y = self.module(x)
            if isinstance(y, (tuple, list)): y = y[0]
            if isinstance(y, dict): y = y["logits"]
            if y.dim() == 3: y = y[:, -1, :]
            if y.shape[-1] != len(self.action_map):
                raise ArtifactError(f"{self.stem}: model output dim {y.shape[-1]} != len(action_map) {len(self.action_map)}")
            p = y if self.output_type == "probabilities" else torch.softmax(y, dim=-1)
            outs.append(p.cpu().numpy())
        p = np.concatenate(outs) if outs else np.zeros((0, len(self.action_map)), np.float32)
        canon = np.zeros((len(p), len(ACTIONS)), np.float32)
        for ci, mi in self.perm.items(): canon[:, ci] = p[:, mi]
        return canon / np.clip(canon.sum(1, keepdims=True), 1e-9, None)

def load_all_policies(d: Path):
    policies, rows, meta = [], [], {"artifact_dir": str(d), "files": {}, "policy_train_end_dates": []}
    for i in range(N_POLICIES):
        f = {k: d / v for k, v in ARTIFACT_FILES[i].items()}
        cfg, schema = _load_json(f["config"]), _load_json(f["schema"])
        cols_s, cols_c = schema.get("feature_columns"), cfg.get("feature_columns")
        cols = cols_s or cols_c
        if not cols: raise ArtifactError(f"{f['schema'].name}: 'feature_columns' not found.")
        if cols_s and cols_c and list(cols_s) != list(cols_c):
            raise ArtifactError(f"{POLICY_NAMES[i]}: feature_columns differ between config and schema files.")
        seq_len = int(schema.get("seq_len") or cfg.get("seq_len") or 0)
        if seq_len < 1: raise ArtifactError(f"{POLICY_NAMES[i]}: 'seq_len' missing in schema/config.")
        raw_map = cfg.get("action_map") or schema.get("action_map")
        if not raw_map: raise ArtifactError(f"{POLICY_NAMES[i]}: 'action_map' missing in config.")
        amap = {int(k): str(v).upper() for k, v in raw_map.items()}
        if not set(amap.values()) <= set(ACTIONS) or sorted(amap) != list(range(len(amap))):
            raise ArtifactError(f"{POLICY_NAMES[i]}: action_map {amap} must map 0..n-1 to a subset of {ACTIONS}.")
        scaler = None
        if str(cfg.get("preprocessing", "sklearn_pickle")).lower() != "none":
            try: scaler = joblib.load(f["preprocessor"])
            except Exception:
                with open(f["preprocessor"], "rb") as fh: scaler = pickle.load(fh)
            nfi = getattr(scaler, "n_features_in_", len(cols))
            if nfi != len(cols):
                raise ArtifactError(f"{POLICY_NAMES[i]}: preprocessor expects {nfi} features, schema lists {len(cols)}.")
        default_arch = {"input_dim": len(cols), "n_actions": len(amap), "seq_len": seq_len}
        module, kind = load_policy_module(f["model"], cfg, default_arch)
        pol = FrozenPolicy(i, ARTIFACT_FILES[i]["model"][:-3], module, kind, cols, seq_len, amap, scaler,
                           str(cfg.get("input_mode", "sequence")).lower(), str(cfg.get("output_type", "logits")).lower(), f)
        try:   # dry-run forward pass of the right shape
            _ = pol.predict_proba(np.zeros((2, seq_len, len(cols)), np.float32))
        except ArtifactError: raise
        except Exception as e:
            raise ArtifactError(f"{POLICY_NAMES[i]}: forward pass failed on (2,{seq_len},{len(cols)}) input: {e}")
        policies.append(pol)
        if cfg.get("train_end_date"): meta["policy_train_end_dates"].append(str(cfg["train_end_date"]))
        meta["files"][pol.stem] = {k: {"name": v.name, "sha256": sha256_of(v)} for k, v in f.items()}
        rows.append(dict(policy=POLICY_NAMES[i], loader=kind, n_features=len(cols), seq_len=seq_len,
                         actions="/".join(amap[j] for j in sorted(amap)), params=sum(p.numel() for p in module.parameters()),
                         sha256=meta["files"][pol.stem]["model"]["sha256"][:12]))
    ref = policies[0]
    for p in policies[1:]:
        if p.columns != ref.columns: raise ArtifactError(f"{POLICY_NAMES[p.idx]}: feature column order/content differs from {POLICY_NAMES[0]}.")
        if p.seq_len != ref.seq_len: raise ArtifactError(f"{POLICY_NAMES[p.idx]}: seq_len {p.seq_len} != {ref.seq_len}.")
        if p.action_map != ref.action_map: raise ArtifactError(f"{POLICY_NAMES[p.idx]}: action_map {p.action_map} != {ref.action_map}.")
    return policies, pd.DataFrame(rows), meta

ARTIFACT_DIR = ensure_artifacts(CFG.paths)
POLICIES, policy_table, ARTIFACT_META = load_all_policies(ARTIFACT_DIR)
SEQ_LEN, POLICY_COLUMNS = POLICIES[0].seq_len, POLICIES[0].columns
print("All five specialist policies loaded, frozen and mutually compatible.")
display(policy_table) if "display" in globals() else print(policy_table)
print(f"Shared schema: {len(POLICY_COLUMNS)} features, seq_len={SEQ_LEN}, action_map={POLICIES[0].action_map}")

## 3. Data loading and feature construction

All features are **causal** (rolling/expanding windows over past data only). A look-ahead unit test later in the notebook recomputes features on truncated history to prove that the row at time *t* does not change when future data is removed.

The specialist policies need the *same* feature columns they were trained on. Three ways to supply them (checked in this order): `CUSTOM_FEATURE_BUILDER` (paste/import your notebook-1-5 feature function), pre-exported `<TICKER>_policy_features.parquet|csv` files, or columns available in this notebook's built-in market-feature frame. If any required column cannot be produced, the notebook stops with a clear error.

In [ ]:
# ---- 3. Data loading, causal features, policy proposals, shadow performance ----
class FeatureError(RuntimeError):
    pass

CUSTOM_FEATURE_BUILDER: Optional[Callable[[pd.DataFrame], pd.DataFrame]] = None   # df(open,high,low,close,volume) -> DataFrame[POLICY_COLUMNS]

def _clean_ohlcv(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy(); df.columns = [str(c).lower() for c in df.columns]
    if "adj close" in df.columns and "close" not in df.columns: df["close"] = df["adj close"]
    need = ["open", "high", "low", "close", "volume"]
    if any(c not in df.columns for c in need): raise ValueError(f"{name}: need columns {need}, got {list(df.columns)}")
    df = df[need].astype(float)
    df.index = pd.to_datetime(df.index).tz_localize(None); df = df[~df.index.duplicated()].sort_index().dropna()
    if (df[["open", "high", "low", "close"]] <= 0).any().any(): raise ValueError(f"{name}: non-positive prices found")
    return df

def load_ohlcv(symbol: str) -> pd.DataFrame:
    D, cache = CFG.data, Path(CFG.paths.market_data_dir); cache.mkdir(parents=True, exist_ok=True)
    f = cache / f"{symbol}.csv"
    if D.source == "yfinance" and not f.exists():
        import yfinance as yf
        raw = yf.download(symbol, start=D.start, end=D.end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex): raw.columns = raw.columns.get_level_values(0)
        if raw is None or len(raw) == 0: raise ValueError(f"No data returned for {symbol}")
        raw.to_csv(f)
    if not f.exists(): raise FileNotFoundError(f"{f} not found (data source='{D.source}'). Upload OHLCV CSVs to {cache}.")
    raw = pd.read_csv(f, index_col=0, parse_dates=True)
    df = _clean_ohlcv(raw, symbol)
    df = df.loc[D.start:] if D.end is None else df.loc[D.start:D.end]
    if len(df) < 800: raise ValueError(f"{symbol}: only {len(df)} rows; need a longer history.")
    return df

MARKET_FEATURES = ["ret_1", "ret_5", "ret_10", "ret_20", "ret_60", "close_sma20", "close_sma50", "close_sma200",
                   "sma50_sma200", "vol_20", "vol_60", "atr_pct_14", "rsi_14", "vol_chg_5", "vol_z_20",
                   "vol_pct_rank_252", "trend_up", "trend_down", "high_vol", "dd_252",
                   "rel_ret_5", "rel_ret_20", "bench_vol_20"]
FLAG_FEATURES = ["trend_up", "trend_down", "high_vol"]          # binary: not z-scored
SHADOW_METRICS = ["mean_ret_20", "vol_20", "hit_rate_20", "drawdown_60"]

def build_market_frame(df: pd.DataFrame, bench: pd.DataFrame) -> pd.DataFrame:
    c, h, l, v = df["close"], df["high"], df["low"], df["volume"]
    r = c.pct_change(); f = pd.DataFrame(index=df.index)
    for n in (1, 5, 10, 20, 60): f[f"ret_{n}"] = c.pct_change(n)
    sma = {n: c.rolling(n).mean() for n in (20, 50, 200)}
    for n in (20, 50, 200): f[f"close_sma{n}"] = c / sma[n] - 1
    f["sma50_sma200"] = sma[50] / sma[200] - 1
    f["vol_20"], f["vol_60"] = r.rolling(20).std(), r.rolling(60).std()
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    f["atr_pct_14"] = tr.rolling(14).mean() / c
    d = c.diff(); g = d.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean(); ls = (-d.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    with np.errstate(divide="ignore", invalid="ignore"):
        rsi = 100 - 100 / (1 + g / ls)
    f["rsi_14"] = (rsi.fillna(50) - 50) / 50
    f["vol_chg_5"] = np.log((v.rolling(5).mean() + 1) / (v.rolling(20).mean() + 1))
    f["vol_z_20"] = ((v - v.rolling(20).mean()) / v.rolling(20).std().replace(0, np.nan)).clip(-5, 5)
    f["vol_pct_rank_252"] = f["vol_20"].rolling(252, min_periods=126).rank(pct=True)   # past-only percentile
    f["trend_up"] = ((c > sma[200]) & (sma[50] > sma[200])).astype(float).where(sma[200].notna())
    f["trend_down"] = ((c < sma[200]) & (sma[50] < sma[200])).astype(float).where(sma[200].notna())
    f["high_vol"] = (f["vol_pct_rank_252"] > 0.75).astype(float).where(f["vol_pct_rank_252"].notna())
    f["dd_252"] = c / c.rolling(252, min_periods=1).max() - 1
    b = bench["close"].reindex(df.index).ffill()
    f["rel_ret_5"], f["rel_ret_20"] = f["ret_5"] - b.pct_change(5), f["ret_20"] - b.pct_change(20)
    f["bench_vol_20"] = b.pct_change().rolling(20).std()
    return f

def regime_labels(f: pd.DataFrame) -> np.ndarray:
    trend = np.where(f["trend_up"] == 1, "up", np.where(f["trend_down"] == 1, "down", "sideways"))
    vol = np.where(f["high_vol"] == 1, "highvol", "lowvol")
    return np.char.add(np.char.add(trend.astype(str), "_"), vol.astype(str))

def get_policy_feature_matrix(ticker: str, df: pd.DataFrame, mf: pd.DataFrame, columns: List[str]) -> np.ndarray:
    if CUSTOM_FEATURE_BUILDER is not None:
        feats = CUSTOM_FEATURE_BUILDER(df.copy()); src = "CUSTOM_FEATURE_BUILDER"
    else:
        fdir = CFG.paths.policy_features_dir; feats = None; src = None
        if fdir:
            for ext, rd in (("parquet", pd.read_parquet), ("csv", lambda p: pd.read_csv(p, index_col=0, parse_dates=True))):
                p = Path(fdir) / f"{ticker}_policy_features.{ext}"
                if p.exists(): feats, src = rd(p), str(p); break
        if feats is None:
            feats = pd.concat([df[["open", "high", "low", "close", "volume"]], mf], axis=1); src = "built-in market frame"
    feats = feats.reindex(df.index)
    miss = [c for c in columns if c not in feats.columns]
    if miss:
        raise FeatureError(f"{ticker}: cannot build policy feature columns {miss[:8]}{'...' if len(miss) > 8 else ''} from {src}. "
                           "Set CUSTOM_FEATURE_BUILDER to the feature function used in notebooks 1-5, or export "
                           "<TICKER>_policy_features.parquet files and set CFG.paths.policy_features_dir.")
    return feats[columns].to_numpy(np.float64)

def policy_probabilities(policies: List[FrozenPolicy], X: np.ndarray):
    """Frozen-policy proposals for every date t using only rows <= t. Returns probs (T,5,4), confidence (T,5), valid (T,)."""
    T, F = X.shape; L = policies[0].seq_len
    nan_row = np.isnan(X).any(axis=1); cs = np.concatenate([[0], np.cumsum(nan_row)])
    valid = np.zeros(T, bool)
    if T >= L:
        idx = np.arange(L - 1, T); valid[idx] = (cs[idx + 1] - cs[idx + 1 - L]) == 0
    probs = np.full((T, N_POLICIES, len(ACTIONS)), 1.0 / len(ACTIONS), np.float32)
    Xf = np.nan_to_num(X, nan=0.0); vidx = np.where(valid)[0]
    for k, p in enumerate(policies):
        win = np.lib.stride_tricks.sliding_window_view(p.transform(Xf), (L, F))[:, 0]      # window j ends at row j+L-1
        if len(vidx): probs[vidx, k, :] = p.predict_proba(win[vidx - (L - 1)])
    return probs, probs.max(axis=2).astype(np.float32), valid

def shadow_features(open_: np.ndarray, close: np.ndarray, probs: np.ndarray, valid: np.ndarray, env_cfg: EnvConfig, risk_cfg: RiskConfig) -> np.ndarray:
    """Causal 'what would policy k have earned recently?' features. Policy k's own argmax actions are replayed on an
    unconstrained shadow position; the metrics at time t use shadow returns realised up to and including day t."""
    T = len(close); step, cap = risk_cfg.order_fraction, risk_cfg.max_position_size
    cost = (env_cfg.fee_bps + env_cfg.slippage_bps) / 1e4
    gap = np.zeros(T); gap[1:] = open_[1:] / close[:-1] - 1       # close_{t-1} -> open_t
    intra = close / open_ - 1                                      # open_t -> close_t
    out = np.full((T, N_POLICIES, len(SHADOW_METRICS)), np.nan, np.float32)
    for k in range(N_POLICIES):
        a = np.where(valid, probs[:, k, :].argmax(1), HOLD); pos = np.zeros(T); p = 0.0
        for t in range(T):
            if a[t] == BUY: p = min(p + step, cap)
            elif a[t] == SELL: p = max(p - step, 0.0)
            elif a[t] == EXIT: p = 0.0
            pos[t] = p
        p1 = np.concatenate([[0.0], pos[:-1]]); p2 = np.concatenate([[0.0], p1[:-1]])      # pos_{t-1}, pos_{t-2}
        r = p2 * gap + p1 * intra - np.abs(p1 - p2) * cost            # decision at t-1 executes at open_t
        s, inv = pd.Series(r), pd.Series((p1 > 0).astype(float))
        hit = ((s > 0) & (inv > 0)).astype(float).rolling(20).sum() / inv.rolling(20).sum().clip(lower=1)
        eq = (1 + s).cumprod()
        out[:, k, 0], out[:, k, 1], out[:, k, 2] = s.rolling(20).mean().values, s.rolling(20).std().values, hit.values
        out[:, k, 3] = (eq / eq.rolling(60, min_periods=1).max() - 1).values
    return out

@dataclass
class TickerBundle:
    ticker: str; dates: np.ndarray; open: np.ndarray; close: np.ndarray
    market_raw: np.ndarray; probs: np.ndarray; conf: np.ndarray; shadow_raw: np.ndarray
    ann_vol: np.ndarray; regime: np.ndarray; valid_from: int

def build_bundle(ticker: str, df: pd.DataFrame, bench: pd.DataFrame, policies: List[FrozenPolicy],
                 env_cfg: EnvConfig, risk_cfg: RiskConfig) -> TickerBundle:
    mf = build_market_frame(df, bench)
    X = get_policy_feature_matrix(ticker, df, mf, policies[0].columns)
    probs, conf, pvalid = policy_probabilities(policies, X)
    shadow = shadow_features(df["open"].values, df["close"].values, probs, pvalid, env_cfg, risk_cfg)
    market_raw = mf[MARKET_FEATURES].to_numpy(np.float64)
    ok = (~np.isnan(market_raw).any(1)) & pvalid & (~np.isnan(shadow.reshape(len(df), -1)).any(1))
    bad = np.where(~ok)[0]; valid_from = int(bad[-1] + 1) if len(bad) else 0
    return TickerBundle(ticker, df.index.values.astype("datetime64[ns]"), df["open"].values, df["close"].values,
                        market_raw, probs, conf, shadow, (mf["vol_20"] * np.sqrt(252)).values, regime_labels(mf), valid_from)

BENCH_DF = load_ohlcv(CFG.data.benchmark)
BUNDLES: Dict[str, TickerBundle] = {}
for tk in CFG.data.tickers:
    BUNDLES[tk] = build_bundle(tk, load_ohlcv(tk), BENCH_DF, POLICIES, CFG.env, CFG.risk)
    b = BUNDLES[tk]; print(f"{tk}: {len(b.dates)} days {str(b.dates[0])[:10]} -> {str(b.dates[-1])[:10]} | first usable index {b.valid_from} ({str(b.dates[min(b.valid_from, len(b.dates)-1)])[:10]})")

## 4. Observation construction

The observation (fixed schema, saved to `meta_rl_observation_schema.json`) concatenates four blocks:

| Block | Content |
|---|---|
| **A. Market context** | multi-horizon returns, MA relationships, volatility/ATR, RSI, volume change, regime flags, benchmark-relative moves. Z-scored with **train-split statistics only**, clipped to ±5 |
| **B. Trader behaviour** | action frequencies, avg holding duration, typical size, trade frequency, reaction to volatility and drawdowns. *Synthetic unless a real `trader_history_csv` is given.* Optional embedding only if a real encoder file exists |
| **C. Investor & position** | risk tolerance, horizon, exposure, unrealised P&L, holding duration, distance to review checkpoint / horizon end, drawdown, last-day return |
| **D. Candidate policies** | action probabilities and confidence of each of the five policies; **shadow performance** of each policy's recent proposals (computed only from data up to *t*) |

No realised future return is ever part of an observation.

In [ ]:
# ---- 4a. Investor profiles (SYNTHETIC unless real history is supplied) ----
@dataclass
class InvestorProfile:
    profile_id: str
    archetype: str
    source: str                      # "synthetic" | "real_trade_history"
    risk_tolerance: float            # 0..1 declared tolerance
    horizon_days: int
    max_drawdown_tol: float          # declared maximum acceptable portfolio drawdown
    max_position_cap: float          # declared maximum single-asset exposure
    action_freq: List[float]         # historical frequency of HOLD, BUY, SELL, EXIT (sums to 1)
    avg_hold_days: float
    typical_position_size: float
    trade_frequency: float           # trades per trading day
    vol_reaction: float              # [-1,1]: (trade rate in high-vol - in low-vol) / (sum); negative = trades less when volatile
    drawdown_reaction: float         # [-1,1]: tendency to cut risk during drawdowns
    authorized_actions: List[str] = field(default_factory=lambda: list(ACTIONS))
    def behavior_vector(self) -> np.ndarray:
        return np.array(list(self.action_freq) + [min(self.avg_hold_days / 60.0, 3.0), self.typical_position_size,
                        self.trade_frequency, self.vol_reaction, self.drawdown_reaction], dtype=np.float32)

BEHAVIOR_FEATURES = ["freq_HOLD", "freq_BUY", "freq_SELL", "freq_EXIT", "avg_hold_days_scaled",
                     "typical_position_size", "trade_frequency", "vol_reaction", "drawdown_reaction"]
ARCHETYPES = {
    "conservative": dict(risk=0.20, dd=0.08, cap=0.15, horizon=504, freq=[0.88, 0.05, 0.04, 0.03], hold=45, size=0.08, tf=0.06, vr=-0.5, dr=0.6),
    "moderate":     dict(risk=0.50, dd=0.15, cap=0.25, horizon=252, freq=[0.80, 0.09, 0.06, 0.05], hold=25, size=0.15, tf=0.12, vr=-0.1, dr=0.3),
    "aggressive":   dict(risk=0.80, dd=0.25, cap=0.30, horizon=126, freq=[0.65, 0.18, 0.09, 0.08], hold=10, size=0.25, tf=0.30, vr=0.3, dr=-0.1),
}

def generate_synthetic_profiles(n_per_archetype: int, seed: int, prefix: str) -> List[InvestorProfile]:
    """SYNTHETIC profiles: archetype templates plus noise. They are experimental proxies, NOT learned from any real person."""
    rng, out = np.random.default_rng(seed), []
    for arch, a in ARCHETYPES.items():
        for j in range(n_per_archetype):
            out.append(InvestorProfile(
                profile_id=f"{prefix}_{arch[:4]}_{j}", archetype=arch, source="synthetic",
                risk_tolerance=float(np.clip(a["risk"] + rng.normal(0, 0.05), 0.05, 0.95)),
                horizon_days=int(max(21, a["horizon"] * rng.uniform(0.8, 1.2))),
                max_drawdown_tol=float(np.clip(a["dd"] * rng.uniform(0.85, 1.15), 0.03, 0.5)),
                max_position_cap=float(np.clip(a["cap"] * rng.uniform(0.85, 1.0), 0.05, CFG.risk.max_position_size)),
                action_freq=[float(x) for x in rng.dirichlet(np.array(a["freq"]) * 60)],
                avg_hold_days=float(a["hold"] * rng.uniform(0.7, 1.3)),
                typical_position_size=float(np.clip(a["size"] * rng.uniform(0.7, 1.3), 0.02, 0.5)),
                trade_frequency=float(np.clip(a["tf"] * rng.uniform(0.7, 1.3), 0.01, 0.9)),
                vol_reaction=float(np.clip(a["vr"] + rng.normal(0, 0.1), -1, 1)),
                drawdown_reaction=float(np.clip(a["dr"] + rng.normal(0, 0.1), -1, 1))))
    return out

def profile_from_trade_history(df: pd.DataFrame, risk_tolerance: float, horizon_days: int, max_drawdown_tol: float,
                               max_position_cap: float, profile_id: str = "real_0") -> InvestorProfile:
    """Real behaviour statistics from an actual trade log with columns: date, action (BUY/SELL/EXIT), size (fraction of equity),
    optional boolean columns high_vol and in_drawdown. Declared risk limits must be supplied by the investor."""
    d = df.copy(); d["date"] = pd.to_datetime(d["date"]); d["action"] = d["action"].str.upper()
    days = max(int((d["date"].max() - d["date"].min()).days * 252 / 365), 1)
    counts = d["action"].value_counts().reindex(["BUY", "SELL", "EXIT"]).fillna(0).values
    freq = np.concatenate([[max(days - counts.sum(), 0)], counts]).astype(float); freq = freq / freq.sum()
    holds, entry = [], None
    for _, r in d.sort_values("date").iterrows():
        if r["action"] == "BUY" and entry is None: entry = r["date"]
        elif r["action"] == "EXIT" and entry is not None: holds.append((r["date"] - entry).days * 252 / 365); entry = None
    def _react(col):
        if col not in d.columns: print(f"  note: '{col}' not in trade history -> reaction feature set to 0.0"); return 0.0
        a, b = d[col].astype(bool).mean(), 1 - d[col].astype(bool).mean(); return float(np.clip((a - b) / max(a + b, 1e-9), -1, 1))
    return InvestorProfile(profile_id, "real", "real_trade_history", risk_tolerance, horizon_days, max_drawdown_tol, max_position_cap,
                           [float(x) for x in freq], float(np.mean(holds)) if holds else 20.0,
                           float(d["size"].median()) if "size" in d.columns else 0.1, float(len(d) / days),
                           _react("high_vol"), _react("in_drawdown"))

PROFILES_TRAIN = generate_synthetic_profiles(CFG.profiles.n_train_per_archetype, CFG.profiles.seeds[0], "tr")
PROFILES_VAL = generate_synthetic_profiles(CFG.profiles.n_val_per_archetype, CFG.profiles.seeds[1], "va")
PROFILES_TEST = generate_synthetic_profiles(CFG.profiles.n_test_per_archetype, CFG.profiles.seeds[2], "te")
if CFG.paths.trader_history_csv:
    print("Real trade history supplied -> adding ONE real-history profile to the TEST set only.")
    print("Declared risk limits below are placeholders: replace them with the investor's actual declared limits.")
    PROFILES_TEST.append(profile_from_trade_history(pd.read_csv(CFG.paths.trader_history_csv), 0.5, 252, 0.15, 0.25))
print(f"Profiles -> train: {len(PROFILES_TRAIN)}, val: {len(PROFILES_VAL)}, test: {len(PROFILES_TEST)} "
      f"(sources: {sorted({p.source for p in PROFILES_TRAIN + PROFILES_VAL + PROFILES_TEST})})")
print("NOTE: synthetic profiles are experimental proxies; results say nothing about any real individual's behaviour.")

# Optional behavioural embedding: used ONLY if a real trained encoder file exists.
BEHAVIOR_ENCODER = None
if CFG.paths.behavior_encoder_path and Path(CFG.paths.behavior_encoder_path).exists():
    BEHAVIOR_ENCODER = torch.jit.load(CFG.paths.behavior_encoder_path, map_location="cpu").eval()
    print("Behaviour encoder loaded (embeddings are NOT wired into the observation by default; extend the schema if you enable them).")
else:
    print("No trained behaviour encoder found -> behavioural embeddings are NOT used (raw behaviour statistics only).")

## 6. Risk controller (deterministic, separate from RL)

Defined before the environment because the environment calls it on every step. The gate checks, in order: invalid action → authorisation (kill switch, mode, per-investor allowed actions) → state validity → drawdown circuit breaker (may **override** to EXIT) → daily-loss limit → volatility restriction → position/exposure limits (may **clip** the order) → available cash. Every non-trivial decision carries a reason code and is logged.

**Transition rule for rejected actions (used consistently everywhere):** a rejected proposal is replaced by HOLD (no trade); an override replaces it by EXIT; a clipped BUY/SELL executes at the reduced size. Rejections and overrides incur the `w_reject` reward penalty so the agent learns to avoid selecting policies whose proposals are likely to be blocked.

In [ ]:
# ---- 6. Deterministic risk controller ----
@dataclass
class RiskState:
    position_frac: float = 0.0
    cash_frac: float = 1.0
    equity: float = 100_000.0
    peak_equity: float = 100_000.0
    drawdown: float = 0.0
    day_return: float = 0.0
    ann_vol: float = 0.0
    mode: str = "simulation"

@dataclass
class RiskDecision:
    approved: bool
    final_action: int
    proposed_action: int
    reason_code: str
    reason: str
    order_fraction: float = 0.0
    override: bool = False
    clipped: bool = False
    @property
    def rejected(self) -> bool: return not self.approved

class RiskController:
    """Pure function of (proposal, state, profile, config). No learning, no randomness, no access to PPO output."""
    def __init__(self, cfg: RiskConfig): self.cfg = cfg
    def _rej(self, a, code, msg): return RiskDecision(False, HOLD, a, code, msg)
    def validate(self, proposed, s: RiskState, profile: InvestorProfile) -> RiskDecision:
        c = self.cfg
        if isinstance(proposed, bool) or not isinstance(proposed, (int, np.integer)) or int(proposed) not in (HOLD, BUY, SELL, EXIT):
            return RiskDecision(False, HOLD, -1, "INVALID_ACTION", f"proposed action {proposed!r} is not in {ACTIONS}")
        a = int(proposed)
        if not c.trading_enabled: return self._rej(a, "KILL_SWITCH", "trading_enabled is False")
        if s.mode not in c.allowed_modes: return self._rej(a, "UNAUTHORIZED_MODE", f"mode '{s.mode}' not in {list(c.allowed_modes)}")
        if ACTIONS[a] not in profile.authorized_actions: return self._rej(a, "UNAUTHORIZED_ACTION", f"{ACTIONS[a]} not authorised for {profile.profile_id}")
        pos = s.position_frac
        dd_limit = min(c.max_drawdown, profile.max_drawdown_tol)
        if s.drawdown >= dd_limit:
            if pos > 1e-9 and a != EXIT:
                return RiskDecision(False, EXIT, a, "DRAWDOWN_BREAKER", f"drawdown {s.drawdown:.1%} >= limit {dd_limit:.1%}: forced EXIT", order_fraction=pos, override=True)
            if a == BUY: return self._rej(a, "DRAWDOWN_BREAKER", f"drawdown {s.drawdown:.1%} >= limit {dd_limit:.1%}: no new risk")
        if a == HOLD: return RiskDecision(True, HOLD, a, "OK", "hold")
        if a in (SELL, EXIT) and pos <= 1e-9: return self._rej(a, "NO_POSITION", f"{ACTIONS[a]} proposed with no open position")
        if a == EXIT: return RiskDecision(True, EXIT, a, "OK", "exit approved", order_fraction=pos)
        if a == SELL:
            frac = min(c.order_fraction, pos); return RiskDecision(True, SELL, a, "OK", "reduce approved", order_fraction=frac)
        # BUY
        if s.day_return <= -c.max_daily_loss: return self._rej(a, "DAILY_LOSS_LIMIT", f"daily return {s.day_return:.2%} <= -{c.max_daily_loss:.2%}")
        vol_limit = c.max_ann_vol_new_buys * (0.6 + 0.8 * profile.risk_tolerance)
        if s.ann_vol > vol_limit: return self._rej(a, "VOLATILITY", f"annualised vol {s.ann_vol:.1%} > limit {vol_limit:.1%}")
        cap = min(c.max_position_size, c.max_portfolio_exposure, profile.max_position_cap); headroom = cap - pos
        if headroom < c.min_order_fraction: return self._rej(a, "MAX_POSITION", f"position {pos:.1%} at cap {cap:.1%}")
        if s.cash_frac < c.min_order_fraction: return self._rej(a, "INSUFFICIENT_CAPITAL", f"cash {s.cash_frac:.2%} below minimum order")
        frac = min(c.order_fraction, headroom, s.cash_frac); clipped = frac < c.order_fraction - 1e-12
        return RiskDecision(True, BUY, a, "CLIPPED_TO_LIMIT" if clipped else "OK", "buy clipped to limit" if clipped else "buy approved", order_fraction=frac, clipped=clipped)

# ---- Unit tests: every rejection path must fire deterministically ----
def _risk_unit_tests():
    rc, prof = RiskController(RiskConfig()), generate_synthetic_profiles(1, 7, "ut")[1]
    S = RiskState
    cases = [
        ("INVALID_ACTION", 9, S()), ("INVALID_ACTION", None, S()), ("INVALID_ACTION", float("nan"), S()),
        ("UNAUTHORIZED_MODE", BUY, S(mode="live")), ("NO_POSITION", SELL, S()), ("NO_POSITION", EXIT, S()),
        ("DAILY_LOSS_LIMIT", BUY, S(day_return=-0.05)), ("VOLATILITY", BUY, S(ann_vol=5.0)),
        ("MAX_POSITION", BUY, S(position_frac=1.0, cash_frac=0.0 + 0.5)), ("INSUFFICIENT_CAPITAL", BUY, S(cash_frac=0.001)),
        ("DRAWDOWN_BREAKER", BUY, S(drawdown=0.9)), ("DRAWDOWN_BREAKER", BUY, S(drawdown=0.9, position_frac=0.1)),
    ]
    for code, act, st in cases:
        d = rc.validate(act, st, prof); assert d.reason_code == code and d.rejected, (code, act, d)
    assert rc.validate(BUY, S(drawdown=0.9, position_frac=0.1), prof).override and rc.validate(BUY, S(drawdown=0.9, position_frac=0.1), prof).final_action == EXIT
    assert rc.validate(BUY, S(position_frac=prof.max_position_cap - 0.01), prof).clipped
    ks = RiskController(RiskConfig(trading_enabled=False)); assert ks.validate(BUY, S(), prof).reason_code == "KILL_SWITCH"
    prof2 = copy.deepcopy(prof); prof2.authorized_actions = ["HOLD", "EXIT"]; assert rc.validate(BUY, S(), prof2).reason_code == "UNAUTHORIZED_ACTION"
    assert rc.validate(BUY, S(), prof).approved and rc.validate(HOLD, S(), prof).approved
    print(f"Risk controller unit tests passed ({len(cases) + 6} checks).")
_risk_unit_tests()

In [ ]:
# ---- 5a. Portfolio accounting and simulated execution (kept separate from selection, proposal and risk code) ----
@dataclass
class Portfolio:
    cash: float
    shares: float = 0.0
    avg_entry: float = 0.0
    hold_days: int = 0
    peak: float = 0.0
    def equity(self, price: float) -> float: return self.cash + self.shares * price

def execute_order(pf: Portfolio, decision: RiskDecision, equity_t: float, open_next: float, env_cfg: EnvConfig) -> Dict[str, float]:
    """Execute an *approved* decision at the next bar's open with slippage and fees. Fractional shares, long-only, no leverage."""
    a, fee, slip = decision.final_action, env_cfg.fee_bps / 1e4, env_cfg.slippage_bps / 1e4
    res = {"notional": 0.0, "cost": 0.0, "shares_delta": 0.0}
    if a == HOLD or (a != EXIT and decision.order_fraction <= 0): return res
    if a == BUY:
        px = open_next * (1 + slip); notional = min(decision.order_fraction * equity_t, pf.cash / (1 + fee))
        if notional <= 1e-9: return res
        sh = notional / px; new_sh = pf.shares + sh
        pf.avg_entry = (pf.avg_entry * pf.shares + px * sh) / new_sh; pf.shares = new_sh
        c_fee = notional * fee; pf.cash -= notional + c_fee
        res.update(notional=notional, cost=c_fee + sh * (px - open_next), shares_delta=sh)
    elif pf.shares > 0:
        px = open_next * (1 - slip)
        sh = pf.shares if a == EXIT else min(pf.shares, decision.order_fraction * equity_t / open_next)
        proceeds = sh * px; c_fee = proceeds * fee; pf.cash += proceeds - c_fee; pf.shares -= sh
        if pf.shares < 1e-9: pf.shares, pf.avg_entry = 0.0, 0.0
        res.update(notional=proceeds, cost=c_fee + sh * (open_next - px), shares_delta=-sh)
    return res
print("Execution model: decision at close(t) -> fill at open(t+1) +/- slippage, per-side fee; long-only, fractional shares, no leverage.")

In [ ]:
# ---- 4b. Chronological splits, normaliser, observation layout, shared observation assembly ----
POSITION_FEATURES = ["position_frac", "unrealized_pnl", "holding_days_scaled", "dist_to_checkpoint", "horizon_remaining",
                     "drawdown", "risk_tolerance", "horizon_years", "last_day_return_scaled"]

def candidate_names(perf: bool) -> List[str]:
    n = [f"p{k}_prob_{a}" for k in range(N_POLICIES) for a in ACTIONS] + [f"p{k}_confidence" for k in range(N_POLICIES)]
    if perf: n += [f"p{k}_shadow_{m}" for k in range(N_POLICIES) for m in SHADOW_METRICS]
    return n

def observation_names(use_profile: bool, use_perf: bool) -> List[Tuple[str, str]]:
    names = [("market", n) for n in MARKET_FEATURES]
    if use_profile: names += [("trader_behavior", n) for n in BEHAVIOR_FEATURES]
    names += [("investor_position", n) for n in POSITION_FEATURES]
    names += [("candidate_policies", n) for n in candidate_names(use_perf)]
    return names

def position_vector(position_frac, unrealized, hold_days, drawdown, last_ret, step_idx, profile: InvestorProfile, checkpoint_days: int) -> np.ndarray:
    dist_cp = (checkpoint_days - (step_idx % checkpoint_days)) / checkpoint_days
    horizon_rem = float(np.clip((profile.horizon_days - step_idx) / profile.horizon_days, 0.0, 1.0))
    return np.array([position_frac, np.clip(unrealized, -1, 1), min(hold_days / 60.0, 3.0), dist_cp, horizon_rem,
                     drawdown, profile.risk_tolerance, profile.horizon_days / 252.0, np.clip(last_ret * 20, -3, 3)], dtype=np.float32)

def assemble_observation(market_n, behavior, pos_vec, core_n, perf_n, use_profile: bool, use_perf: bool) -> np.ndarray:
    parts = [market_n] + ([behavior] if use_profile else []) + [pos_vec, core_n] + ([perf_n] if use_perf else [])
    return np.clip(np.concatenate(parts), -10, 10).astype(np.float32)

class Normalizer:
    """Z-score with TRAIN-split statistics for market and shadow-performance features; flags/probabilities are left as is."""
    def __init__(self): self.m_mean = self.m_std = self.s_mean = self.s_std = None; self.clip = 5.0
    def fit(self, market_rows: np.ndarray, shadow_rows: np.ndarray):
        self.m_mean, self.m_std = np.nanmean(market_rows, 0), np.nanstd(market_rows, 0)
        for j, n in enumerate(MARKET_FEATURES):
            if n in FLAG_FEATURES: self.m_mean[j], self.m_std[j] = 0.0, 1.0
        self.s_mean, self.s_std = np.nanmean(shadow_rows, 0), np.nanstd(shadow_rows, 0)
        self.m_std, self.s_std = np.maximum(self.m_std, 1e-8), np.maximum(self.s_std, 1e-8); return self
    def market(self, x): return np.nan_to_num(np.clip((x - self.m_mean) / self.m_std, -self.clip, self.clip))
    def shadow(self, x): return np.nan_to_num(np.clip((x - self.s_mean) / self.s_std, -self.clip, self.clip))
    def to_dict(self): return dict(market_mean=self.m_mean.tolist(), market_std=self.m_std.tolist(), shadow_mean=self.s_mean.tolist(), shadow_std=self.s_std.tolist(), clip=self.clip)
    @classmethod
    def from_dict(cls, d):
        n = cls(); n.m_mean, n.m_std, n.s_mean, n.s_std = [np.array(d[k], dtype=float) for k in ("market_mean", "market_std", "shadow_mean", "shadow_std")]
        n.clip = float(d["clip"]); return n

class DataStore:
    def __init__(self, bundles: Dict[str, TickerBundle], dcfg: DataConfig):
        self.bundles, self.tickers, self.norm, self._cache = bundles, list(bundles), None, {}
        self.split_idx: Dict[str, Dict[str, Tuple[int, int]]] = {"train": {}, "val": {}, "test": {}}
        e, self.split_dates = dcfg.embargo_days, {}
        for tk, b in bundles.items():
            d = b.dates; tr_end = int(np.searchsorted(d, np.datetime64(dcfg.train_end), side="right") - 1)
            va_start = tr_end + 1 + e; va_end = int(np.searchsorted(d, np.datetime64(dcfg.val_end), side="right") - 1)
            te_start = va_end + 1 + e; te_end = len(d) - 1
            spans = {"train": (b.valid_from, tr_end), "val": (va_start, va_end), "test": (te_start, te_end)}
            for sp, (i0, i1) in spans.items():
                if i1 - i0 < dcfg.min_split_days: raise ValueError(f"{tk}: {sp} split has only {i1 - i0} usable days; adjust dates/history.")
                self.split_idx[sp][tk] = (i0, i1)
            self.split_dates[tk] = {sp: (str(d[i0])[:10], str(d[i1])[:10]) for sp, (i0, i1) in spans.items()}
    def fit_normalizer(self):
        m = np.concatenate([self.bundles[t].market_raw[i0:i1 + 1] for t, (i0, i1) in self.split_idx["train"].items()])
        s = np.concatenate([self.bundles[t].shadow_raw[i0:i1 + 1].reshape(i1 - i0 + 1, -1) for t, (i0, i1) in self.split_idx["train"].items()])
        self.norm = Normalizer().fit(m, s); self._cache = {}; return self.norm
    def arrays(self, ticker: str):
        if ticker not in self._cache:
            b = self.bundles[ticker]; T = len(b.close)
            self._cache[ticker] = dict(market=self.norm.market(b.market_raw).astype(np.float32),
                                       perf=self.norm.shadow(b.shadow_raw.reshape(T, -1)).astype(np.float32),
                                       core=np.concatenate([b.probs.reshape(T, -1), b.conf], axis=1).astype(np.float32))
        return self._cache[ticker]

STORE = DataStore(BUNDLES, CFG.data)
STORE.fit_normalizer()
print(pd.DataFrame({tk: {sp: f"{a} -> {b}" for sp, (a, b) in v.items()} for tk, v in STORE.split_dates.items()}).T)

# ---- Leakage guard against the specialist policies' own training period ----
if ARTIFACT_META["policy_train_end_dates"]:
    pol_end = max(pd.Timestamp(x) for x in ARTIFACT_META["policy_train_end_dates"])
    test_start = min(pd.Timestamp(STORE.split_dates[t]["test"][0]) for t in STORE.tickers)
    msg = f"Specialist policies were trained up to {pol_end.date()}, but the Meta-RL test period starts {test_start.date()}."
    if test_start <= pol_end:
        if CFG.data.strict_leakage_check: raise RuntimeError(msg + " The held-out test would be contaminated. Move CFG.data.train_end/val_end later, or re-split.")
        print("WARNING:", msg)
    else: print("Leakage check passed:", msg)
else:
    print("WARNING: policy configs carry no 'train_end_date', so overlap between the specialists' training data and this notebook's "
          "validation/test periods cannot be verified automatically. Confirm manually that the test period postdates all policy training data.")

## 5. Custom Gymnasium environment

Per step: build observation (past data only) → PPO picks a policy → that frozen policy's argmax action is the **proposal** → deterministic risk gate → fill at next open with fees/slippage → mark to next close → reward.

**Reward** (all weights in `RewardConfig`):

`r = w_perf·DSR − w_dd·(excess_drawdown/tol + 10·new_drawdown_increase) − w_cost·cost_bps/10 − w_turnover·|Δposition| + w_behavior·(freq_profile[a_exec] − 0.25) − w_reject·1[rejected or overridden]`

* **DSR** is the differential Sharpe ratio (Moody & Saffell): an online risk-adjusted performance measure of the *realised portfolio return of the executed action* (mean-variance utility is available as an alternative).
* **Behavioural term:** deliberately small and **not** treated as profitability. Behavioural consistency, risk compliance and financial performance are reported as three separate quantities.
* The reward uses only the realised outcome of the action that was actually executed. There is no hindsight "best policy" signal, no future-price matching, and no future return in any observation.

In [ ]:
# ---- 5b. Gymnasium environment ----
class MetaRLTradingEnv(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, store: DataStore, split: str, profiles: List[InvestorProfile], env_cfg: EnvConfig, reward_cfg: RewardConfig,
                 risk_cfg: RiskConfig, mode: str = "random", record: bool = False, tickers: Optional[List[str]] = None):
        super().__init__()
        self.store, self.split, self.profiles, self.ec, self.rc, self.risk_cfg = store, split, list(profiles), env_cfg, reward_cfg, risk_cfg
        self.risk, self.mode, self.record = RiskController(risk_cfg), mode, record
        self.tickers = list(tickers or store.tickers); self.windows = {t: store.split_idx[split][t] for t in self.tickers}
        self.combos = [(t, p) for t in self.tickers for p in self.profiles]
        self.use_profile, self.use_perf = env_cfg.use_trader_profile, env_cfg.use_candidate_perf
        self.obs_names = observation_names(self.use_profile, self.use_perf)
        self.observation_space = spaces.Box(-10.0, 10.0, (len(self.obs_names),), np.float32)
        self.action_space = spaces.Discrete(N_POLICIES)
        self._cursor, self.pending_probs, self.t = 0, None, 0

    # ---------- helpers ----------
    def current_regime(self) -> str: return str(self.b.regime[self.t])
    def _position_features(self) -> np.ndarray:
        price = self.b.close[self.t]; eq = self.pf.equity(price)
        unreal = (price / self.pf.avg_entry - 1.0) if self.pf.shares > 0 else 0.0
        return position_vector(self.pf.shares * price / eq, unreal, self.pf.hold_days, 1 - eq / self.pf.peak, self.last_ret,
                               self.t - self.start, self.profile, self.ec.checkpoint_days)
    def _obs(self) -> np.ndarray:
        a, t = self.arr, self.t
        return assemble_observation(a["market"][t], self.profile.behavior_vector(), self._position_features(), a["core"][t], a["perf"][t], self.use_profile, self.use_perf)

    # ---------- gym API ----------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); options = options or {}
        if "ticker" in options: ticker, profile = options["ticker"], options["profile"]
        elif self.mode == "sequential": ticker, profile = self.combos[self._cursor % len(self.combos)]; self._cursor += 1
        else: ticker, profile = self.tickers[self.np_random.integers(len(self.tickers))], self.profiles[self.np_random.integers(len(self.profiles))]
        i0, i1 = self.windows[ticker]
        if self.mode == "random" and "ticker" not in options:
            L = min(self.ec.episode_len, i1 - i0); self.start = int(self.np_random.integers(i0, i1 - L + 1)); self.end = self.start + L
        else: self.start, self.end = i0, i1
        self.ticker, self.profile, self.b, self.arr = ticker, profile, self.store.bundles[ticker], self.store.arrays(ticker)
        self.t = self.start; self.pf = Portfolio(cash=self.ec.initial_capital, peak=self.ec.initial_capital)
        self.last_ret, self.dd_prev, self.A, self.B, self.n_dsr = 0.0, 0.0, 0.0, 0.0, 0
        self.exec_counts, self.hold_done, self.rej_n, self.override_n, self.trade_n = np.zeros(4), [], 0, 0, 0
        self.reward_parts = {k: 0.0 for k in ("perf", "dd", "cost", "turnover", "behavior", "reject")}
        self.steps: List[dict] = []; self.risk_log: List[dict] = []
        self.eq_hist = [self.ec.initial_capital]; self.date_hist = [self.b.dates[self.t]]
        return self._obs(), {}

    def _dsr(self, r: float) -> float:
        eta, dA, dB = self.rc.dsr_eta, r - self.A, r * r - self.B; var = self.B - self.A ** 2
        d = 0.0 if (var <= 1e-12 or self.n_dsr < self.rc.dsr_warmup) else (self.B * dA - 0.5 * self.A * dB) / (var ** 1.5)
        self.A += eta * dA; self.B += eta * dB; self.n_dsr += 1
        return float(np.clip(d, -self.rc.dsr_clip, self.rc.dsr_clip))

    def set_selector_probs(self, p): self.pending_probs = None if p is None else np.asarray(p, float)

    def step(self, action):
        k = int(action); proposed = int(np.argmax(self.b.probs[self.t, k]))      # selected frozen policy's actual proposal
        return self._transition(proposed, k)

    def step_proposal(self, proposed: int, selected: int = -1):
        """Used by non-RL baselines: same gate, execution and accounting as the RL path."""
        return self._transition(int(proposed), selected)

    def _transition(self, proposed: int, selected: int):
        b, t, pf, rc = self.b, self.t, self.pf, self.rc
        price_c = b.close[t]; eq_t = pf.equity(price_c); pos_before = pf.shares * price_c / eq_t
        state = RiskState(pos_before, pf.cash / eq_t, eq_t, pf.peak, 1 - eq_t / pf.peak, self.last_ret, float(b.ann_vol[t]), self.risk_cfg.mode)
        dec = self.risk.validate(proposed, state, self.profile)
        was_flat_before = pf.shares <= 0
        fill = execute_order(pf, dec, eq_t, b.open[t + 1], self.ec)
        if pf.shares > 0: pf.hold_days += 1
        elif not was_flat_before:
            self.hold_done.append(pf.hold_days); pf.hold_days = 0
        eq_n = pf.equity(b.close[t + 1]); pf.peak = max(pf.peak, eq_n); dd = 1 - eq_n / pf.peak
        ret = eq_n / eq_t - 1.0
        # ---- reward ----
        perf = self._dsr(ret) if rc.perf_mode == "dsr" else (ret * 100 - 0.5 * rc.mv_lambda * (1 - self.profile.risk_tolerance + 0.5) * (ret * 100) ** 2)
        tol = max(self.profile.max_drawdown_tol, 1e-3)
        dd_pen = rc.w_dd * (max(0.0, dd - rc.dd_soft_frac * tol) / tol + 10.0 * max(0.0, dd - self.dd_prev))
        cost_pen = rc.w_cost * (fill["cost"] / eq_t) * 1e3
        turn_pen = rc.w_turnover * (fill["notional"] / eq_t)
        beh = rc.w_behavior * (self.profile.action_freq[dec.final_action] - 0.25)
        rej = rc.w_reject * float(dec.rejected)
        reward = rc.w_perf * perf - dd_pen - cost_pen - turn_pen + beh - rej
        for key, val in (("perf", rc.w_perf * perf), ("dd", -dd_pen), ("cost", -cost_pen), ("turnover", -turn_pen), ("behavior", beh), ("reject", -rej)): self.reward_parts[key] += val
        # ---- bookkeeping ----
        self.exec_counts[dec.final_action] += 1; self.rej_n += int(dec.rejected); self.override_n += int(dec.override); self.trade_n += int(fill["notional"] > 0)
        info = {"selected_policy": selected, "proposed_action": proposed, "final_action": dec.final_action, "risk_reason": dec.reason_code}
        if self.record:
            self.steps.append(dict(date=str(b.dates[t])[:10], ticker=self.ticker, profile_id=self.profile.profile_id, archetype=self.profile.archetype,
                regime=str(b.regime[t]), selected_policy_id=selected, selected_policy_name=(POLICY_NAMES[selected] if selected >= 0 else "n/a (rule baseline)"),
                selector_probs=(None if self.pending_probs is None else np.round(self.pending_probs, 4).tolist()),
                policy_proposals=[ACTIONS[int(np.argmax(b.probs[t, j]))] for j in range(N_POLICIES)],
                policy_confidences=np.round(b.conf[t], 4).tolist(),
                proposed_action=(ACTIONS[proposed] if 0 <= proposed < 4 else str(proposed)), proposed_confidence=(float(b.conf[t, selected]) if selected >= 0 else np.nan),
                final_action=ACTIONS[dec.final_action], risk_approved=dec.approved, risk_override=dec.override, risk_clipped=dec.clipped,
                risk_reason_code=dec.reason_code, risk_reason=dec.reason, position_frac_before=pos_before, equity_before=eq_t,
                traded_notional=fill["notional"], cost=fill["cost"], equity_after=eq_n, ret=ret, drawdown=dd, reward=reward))
            if (not dec.approved) or dec.clipped:
                self.risk_log.append(dict(date=str(b.dates[t])[:10], ticker=self.ticker, profile_id=self.profile.profile_id, selected_policy_id=selected,
                                          proposed_action=(ACTIONS[proposed] if 0 <= proposed < 4 else str(proposed)), final_action=ACTIONS[dec.final_action],
                                          code=dec.reason_code, reason=dec.reason))
            self.eq_hist.append(eq_n); self.date_hist.append(b.dates[t + 1])
        self.last_ret, self.dd_prev, self.t = ret, dd, t + 1
        truncated = self.t >= self.end; self.pending_probs = None
        return self._obs(), float(reward), False, truncated, info

    # ---------- episode-level reporting ----------
    def behavior_stats(self) -> Dict[str, float]:
        n = max(self.exec_counts.sum(), 1); f = self.exec_counts / n
        freq_c = 1.0 - 0.5 * float(np.abs(f - np.array(self.profile.action_freq)).sum())
        holds = list(self.hold_done) + ([self.pf.hold_days] if self.pf.shares > 0 else [])
        hold_c = float(np.exp(-abs(np.log((np.mean(holds) + 1) / (self.profile.avg_hold_days + 1))))) if holds else np.nan
        return {"freq_consistency": freq_c, "hold_compat": hold_c, "behavior_consistency": float(np.nanmean([freq_c, hold_c]))}

    def episode_summary(self) -> dict:
        eq = np.array(self.eq_hist); st = pd.DataFrame(self.steps)
        return dict(ticker=self.ticker, profile=self.profile, equity=eq, dates=np.array(self.date_hist),
                    rets=st["ret"].values if len(st) else np.array([]), regimes=st["regime"].values if len(st) else np.array([]),
                    steps=st, risk_log=pd.DataFrame(self.risk_log), traded=st["traded_notional"].values if len(st) else np.array([]),
                    rejected=int(self.rej_n), overrides=int(self.override_n), trades=int(self.trade_n), decisions=len(st),
                    exec_counts=self.exec_counts.copy(), behavior=self.behavior_stats(), reward_parts=dict(self.reward_parts),
                    total_reward=float(sum(self.reward_parts.values())))

# ---- Gymnasium/SB3 compliance check and a causality test ----
from stable_baselines3.common.env_checker import check_env
_chk = MetaRLTradingEnv(STORE, "train", PROFILES_TRAIN, CFG.env, CFG.reward, CFG.risk)
check_env(_chk, warn=True)
_o, _ = _chk.reset(seed=0)
print(f"check_env passed | observation dim = {_o.shape[0]} | action space = {_chk.action_space}")

In [ ]:
# ---- Look-ahead (causality) test + risk-gate integration test ----
def test_no_lookahead(ticker: str, n_points: int = 3, seed: int = 0):
    """Recompute every feature/proposal/shadow metric on history truncated at t and compare with the full-history row t."""
    df, rng = load_ohlcv(ticker), np.random.default_rng(seed); full = BUNDLES[ticker]
    lo, hi = max(full.valid_from + 5, 400), len(df) - 2
    for t in rng.choice(np.arange(lo, hi), n_points, replace=False):
        sub = build_bundle(ticker, df.iloc[: t + 1], BENCH_DF.loc[: df.index[t]], POLICIES, CFG.env, CFG.risk)
        for name in ("market_raw", "probs", "conf", "shadow_raw", "ann_vol"):
            a, b = getattr(sub, name)[-1], getattr(full, name)[t]
            assert np.allclose(a, b, atol=1e-5, equal_nan=True), f"LOOK-AHEAD LEAK in '{name}' at {df.index[t].date()} for {ticker}"
    print(f"No-look-ahead test passed for {ticker} at {n_points} random dates (market features, policy proposals, shadow metrics).")
test_no_lookahead(CFG.data.tickers[0])

# A hostile selector that always picks the policy proposing BUY must still be stopped by the gate.
def _gate_integration_test():
    env = MetaRLTradingEnv(STORE, "val", PROFILES_VAL, CFG.env, CFG.reward, CFG.risk, mode="sequential", record=True)
    obs, _ = env.reset(); done, max_pos = False, 0.0
    cap = min(CFG.risk.max_position_size, CFG.risk.max_portfolio_exposure, env.profile.max_position_cap)
    while not done:
        buys = [j for j in range(N_POLICIES) if np.argmax(env.b.probs[env.t, j]) == BUY]
        obs, r, term, trunc, info = env.step(buys[0] if buys else 0)
        px = env.b.close[env.t]; max_pos = max(max_pos, env.pf.shares * px / env.pf.equity(px)); done = term or trunc
    s = env.episode_summary()
    print(f"Gate integration test: buy-seeking selector, {s['decisions']} decisions, {s['rejected']} rejected/overridden, "
          f"peak exposure {max_pos:.1%} vs cap {cap:.1%} (drift above cap only from price moves after the order).")
    assert (s["steps"]["position_frac_before"] <= cap + 0.10).all()
_gate_integration_test()

## 7. PPO training

* Stable-Baselines3 PPO, `MlpPolicy`, discrete(5) actions, `DummyVecEnv` with several parallel environments (random ticker, random investor profile, random 252-day window from the **train** split each episode).
* **Evaluation during training** on the chronological **validation** split (deterministic policy, every (ticker × validation-profile) pair once). The best validation checkpoint is kept; training stops early after `patience` validations without improvement (after `min_evals`) or at the timestep budget.
* The **test split is never touched here**, so it is not used for model selection or early stopping.
* Seeds are fixed (`CFG.seed`). PPO has high run-to-run variance; for a paper, repeat with several seeds and report the spread.

In [ ]:
# ---- 7a. Training utilities ----
def make_env(split, profiles, env_cfg, reward_cfg, risk_cfg, mode="random", record=False):
    return MetaRLTradingEnv(STORE, split, profiles, env_cfg, reward_cfg, risk_cfg, mode=mode, record=record)

def run_episode_selector(env, selector, options=None):
    obs, _ = env.reset(options=options); done = False
    while not done:
        a = selector(obs, env); obs, r, term, trunc, info = env.step(a); done = term or trunc
    return env.episode_summary()

def run_episode_rule(env, rule, options=None):
    obs, _ = env.reset(options=options); done = False
    while not done:
        obs, r, term, trunc, info = env.step_proposal(rule(env), selected=-1); done = term or trunc
    return env.episode_summary()

class PPOSelector:
    """Deterministic inference: argmax of the PPO policy distribution (equals model.predict(deterministic=True))."""
    def __init__(self, model): self.model = model
    def probs(self, obs) -> np.ndarray:
        with torch.no_grad():
            t, _ = self.model.policy.obs_to_tensor(np.asarray(obs)[None]); return self.model.policy.get_distribution(t).distribution.probs.cpu().numpy()[0]
    def __call__(self, obs, env=None):
        p = self.probs(obs)
        if env is not None: env.set_selector_probs(p)
        return int(np.argmax(p))

class ValidationCallback(BaseCallback):
    def __init__(self, tag, out_dir, env_cfg, reward_cfg, risk_cfg, ppo_cfg, profiles_val):
        super().__init__(); self.tag, self.out_dir, self.pc = tag, Path(out_dir), ppo_cfg
        self.env = make_env("val", profiles_val, env_cfg, reward_cfg, risk_cfg, mode="sequential", record=True)
        self.history, self.best, self.bad, self._last = [], -np.inf, 0, 0; self.best_step = 0; self.stopped_early = False
    def _validate(self):
        self.env._cursor = 0; rows = []
        for _ in range(len(self.env.combos)):
            s = run_episode_selector(self.env, PPOSelector(self.model)); m = metrics_from_summary(s)
            rows.append(dict(reward=s["total_reward"], sharpe=m["sharpe"], cum_return=m["cum_return"], mdd=m["max_drawdown"], rej=m["rejection_rate"]))
        df = pd.DataFrame(rows); return dict(timesteps=int(self.num_timesteps), mean_reward=df.reward.mean(), mean_sharpe=df.sharpe.mean(),
                                            mean_cum_return=df.cum_return.mean(), mean_mdd=df.mdd.mean(), mean_rejection_rate=df.rej.mean())
    def _on_step(self) -> bool:
        if self.num_timesteps - self._last < self.pc.eval_freq: return True
        self._last = self.num_timesteps; r = self._validate(); self.history.append(r)
        score = r[self.pc.selection_metric]
        if score > self.best:
            self.best, self.bad, self.best_step = score, 0, r["timesteps"]; self.model.save(self.out_dir / "best_model")
        else: self.bad += 1
        print(f"[{self.tag}] t={r['timesteps']:>8,} val reward {r['mean_reward']:+.3f} | Sharpe {r['mean_sharpe']:+.2f} | ret {r['mean_cum_return']:+.2%} | "
              f"rej {r['mean_rejection_rate']:.1%} | best@{self.best_step:,} (bad={self.bad})")
        if len(self.history) >= self.pc.min_evals and self.bad >= self.pc.patience:
            print(f"[{self.tag}] early stop: no validation improvement for {self.bad} evaluations."); self.stopped_early = True; return False
        return True

def train_selector(tag, env_cfg, reward_cfg, risk_cfg, ppo_cfg, total_timesteps=None, seed=None, verbose=True):
    seed = CFG.seed if seed is None else seed; set_global_seed(seed)
    out_dir = OUT / "runs" / tag; out_dir.mkdir(parents=True, exist_ok=True)
    venv = DummyVecEnv([(lambda: Monitor(make_env("train", PROFILES_TRAIN, env_cfg, reward_cfg, risk_cfg, "random"))) for _ in range(ppo_cfg.n_envs)])
    model = PPO("MlpPolicy", venv, learning_rate=ppo_cfg.learning_rate, n_steps=ppo_cfg.n_steps, batch_size=ppo_cfg.batch_size, n_epochs=ppo_cfg.n_epochs,
                gamma=ppo_cfg.gamma, gae_lambda=ppo_cfg.gae_lambda, clip_range=ppo_cfg.clip_range, ent_coef=ppo_cfg.ent_coef, vf_coef=ppo_cfg.vf_coef,
                max_grad_norm=ppo_cfg.max_grad_norm, policy_kwargs=dict(net_arch=dict(pi=list(ppo_cfg.net_arch), vf=list(ppo_cfg.net_arch))),
                seed=seed, device=ppo_cfg.device, verbose=0)
    model.set_logger(sb3_configure(str(out_dir), ["csv"]))
    cb = ValidationCallback(tag, out_dir, env_cfg, reward_cfg, risk_cfg, ppo_cfg, PROFILES_VAL)
    budget = int(total_timesteps or ppo_cfg.total_timesteps); t0 = time.time()
    model.learn(total_timesteps=budget, callback=cb, progress_bar=False)
    model.save(out_dir / "final_model")
    best_path = out_dir / "best_model.zip"
    best = PPO.load(best_path if best_path.exists() else out_dir / "final_model", device="cpu")
    pd.DataFrame(cb.history).to_csv(out_dir / "validation_log.csv", index=False)
    info = dict(tag=tag, seed=seed, budget=budget, trained_timesteps=int(model.num_timesteps), best_val_step=cb.best_step, best_val_score=float(cb.best),
                stopped_early=cb.stopped_early, minutes=round((time.time() - t0) / 60, 2), selection_metric=ppo_cfg.selection_metric,
                had_validation_checkpoint=best_path.exists())
    print(f"[{tag}] done: {info}"); return best, cb.history, info

In [ ]:
# ---- 9a. Metrics (shared by validation, test, baselines and ablations) ----
def perf_from_equity(eq: np.ndarray) -> Dict[str, float]:
    eq = np.asarray(eq, float); rets = eq[1:] / eq[:-1] - 1; n = len(rets); yrs = max(n / 252, 1e-9)
    mu, sd = rets.mean() if n else 0.0, rets.std(ddof=1) if n > 1 else 0.0
    dsd = float(np.sqrt(np.mean(np.minimum(rets, 0) ** 2))) if n else 0.0
    return dict(cum_return=float(eq[-1] / eq[0] - 1), ann_return=float((eq[-1] / eq[0]) ** (1 / yrs) - 1) if eq[-1] > 0 else -1.0,
                sharpe=float(mu / sd * np.sqrt(252)) if sd > 1e-12 else 0.0,
                sortino=float(mu / dsd * np.sqrt(252)) if dsd > 1e-12 else (0.0 if abs(mu) < 1e-12 else np.nan),
                max_drawdown=float((eq / np.maximum.accumulate(eq) - 1).min()))

def metrics_from_summary(s: dict) -> Dict[str, float]:
    m = perf_from_equity(s["equity"]); n = max(len(s["equity"]) - 1, 1); yrs = n / 252
    prof, st = s["profile"], s.get("steps")
    m["turnover_ann"] = float(np.sum(s["traded"]) / np.mean(s["equity"]) / yrs) if len(s.get("traded", [])) else 0.0
    m["trade_count"] = int(s.get("trades", 0)); m["rejection_rate"] = s["rejected"] / max(s["decisions"], 1) if s.get("decisions") else 0.0
    if st is not None and len(st):
        nonhold = (st["proposed_action"] != "HOLD").sum(); m["rejection_rate_nonhold"] = float((~st["risk_approved"]).sum() / max(nonhold, 1))
        m["avg_exposure"] = float(st["position_frac_before"].mean())
        sel = st["selected_policy_id"].values
        for k in range(N_POLICIES): m[f"sel_freq_{k}"] = float((sel == k).mean()) if (sel >= 0).any() else np.nan
    else:
        m["rejection_rate_nonhold"], m["avg_exposure"] = 0.0, np.nan
        for k in range(N_POLICIES): m[f"sel_freq_{k}"] = np.nan
    m["override_count"] = int(s.get("overrides", 0))
    b = s.get("behavior") or {}
    m["behavior_consistency"], m["freq_consistency"], m["hold_compat"] = b.get("behavior_consistency", np.nan), b.get("freq_consistency", np.nan), b.get("hold_compat", np.nan)
    m["investor_dd_breach"] = float(-m["max_drawdown"] > prof.max_drawdown_tol)      # risk compliance (separate from performance)
    m["total_reward"] = s.get("total_reward", np.nan)
    return m

def regime_table(summaries: List[dict], label: str) -> pd.DataFrame:
    rows = []
    for s in summaries:
        if len(s["rets"]): rows.append(pd.DataFrame({"regime": s["regimes"], "ret": s["rets"]}))
    if not rows: return pd.DataFrame()
    d = pd.concat(rows); g = d.groupby("regime")["ret"]
    out = pd.DataFrame({"n_days": g.size(), "mean_daily_ret_bps": g.mean() * 1e4, "ann_sharpe": g.mean() / g.std().replace(0, np.nan) * np.sqrt(252), "hit_rate": g.apply(lambda x: (x > 0).mean())})
    out.insert(0, "strategy", label); return out.reset_index()

In [ ]:
# ---- 7b. Train the full Meta-RL selector ----
print(f"Training budget: {CFG.ppo.total_timesteps:,} timesteps | early stopping: patience={CFG.ppo.patience}, min_evals={CFG.ppo.min_evals} | quick_run={CFG.quick_run}")
MODEL_FULL, HIST_FULL, INFO_FULL = train_selector("full", CFG.env, CFG.reward, CFG.risk, CFG.ppo)

## 8. Validation

Validation is used for model selection and early stopping only. Below: the learning curve and the selector's behaviour on the validation split. **No test data has been touched yet.**

In [ ]:
# ---- 8. Validation report ----
hist = pd.DataFrame(HIST_FULL); display(hist.round(4))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
if len(hist):
    ax[0].plot(hist.timesteps, hist.mean_reward, marker="o"); ax[0].axvline(INFO_FULL["best_val_step"], ls="--", c="g"); ax[0].set_title("Validation mean episode reward")
    ax[1].plot(hist.timesteps, hist.mean_sharpe, marker="o", c="C1"); ax[1].set_title("Validation mean Sharpe")
    ax[2].plot(hist.timesteps, hist.mean_rejection_rate, marker="o", c="C3"); ax[2].set_title("Validation risk-gate rejection rate")
for a in ax: a.set_xlabel("timesteps"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
prog = OUT / "runs" / "full" / "progress.csv"
if prog.exists():
    pr = pd.read_csv(prog)
    if "rollout/ep_rew_mean" in pr:
        plt.figure(figsize=(6, 3)); plt.plot(pr["time/total_timesteps"], pr["rollout/ep_rew_mean"]); plt.title("Training episode reward (rolling mean)"); plt.grid(alpha=.3); plt.show()

# Validation-split evaluation of the chosen checkpoint (deterministic)
_venv = make_env("val", PROFILES_VAL, CFG.env, CFG.reward, CFG.risk, "sequential", record=True)
VAL_SUMMARIES = [run_episode_selector(_venv, PPOSelector(MODEL_FULL)) for _ in range(len(_venv.combos))]
_vm = pd.DataFrame([metrics_from_summary(s) for s in VAL_SUMMARIES])
print("Validation (chosen checkpoint) – mean across ticker x profile runs:")
display(_vm[["cum_return", "sharpe", "sortino", "max_drawdown", "turnover_ann", "trade_count", "rejection_rate", "behavior_consistency"]].mean().round(4).to_frame("mean"))

## 6b. Reward sensitivity (validation data only)

The full reward is a weighted sum of separately measurable terms. Here each variant is retrained at a smaller budget and compared **on the validation split** (not the test split, so the test set stays untouched). Because the variants optimise *different* objectives, compare their risk/behaviour outcomes rather than their raw reward values.

In [ ]:
# ---- Reward sensitivity (validation only) ----
SENS_RESULTS = {}
if CFG.run_sensitivity:
    variants = {"default": {}, "drawdown_heavy": dict(w_dd=6.0), "cost_heavy": dict(w_cost=5.0, w_turnover=3.0),
                "behavior_heavy": dict(w_behavior=0.5), "no_behavior_term": dict(w_behavior=0.0),
                "perf_only": dict(w_dd=0.0, w_cost=0.0, w_turnover=0.0, w_behavior=0.0, w_reject=0.0)}
    for name, ov in variants.items():
        rcfg = copy.deepcopy(CFG.reward)
        for k, v in ov.items(): setattr(rcfg, k, v)
        pc = copy.deepcopy(CFG.ppo); pc.selection_metric = "mean_sharpe"      # comparable selection criterion across different reward scales
        model, hist, info = train_selector(f"sens_{name}", CFG.env, rcfg, CFG.risk, pc, total_timesteps=CFG.ppo.sens_timesteps, verbose=False)
        env = make_env("val", PROFILES_VAL, CFG.env, rcfg, CFG.risk, "sequential", record=True)
        sums = [run_episode_selector(env, PPOSelector(model)) for _ in range(len(env.combos))]
        mm = pd.DataFrame([metrics_from_summary(s) for s in sums]).mean(numeric_only=True)
        SENS_RESULTS[name] = dict(overrides=ov, **{k: float(mm[k]) for k in ("cum_return", "sharpe", "max_drawdown", "turnover_ann", "trade_count", "rejection_rate", "behavior_consistency", "investor_dd_breach")},
                                  **{f"sel_freq_{k}": float(mm[f"sel_freq_{k}"]) for k in range(N_POLICIES)})
    SENS_DF = pd.DataFrame(SENS_RESULTS).T.drop(columns="overrides").astype(float)
    print("Reward sensitivity on the VALIDATION split (retrained at a reduced budget; differences may be within PPO seed noise):")
    display(SENS_DF.round(4))
else:
    SENS_DF = pd.DataFrame(); print("Sensitivity skipped (CFG.run_sensitivity=False).")

## 10a. Ablations (train the variants on the same data, seed and budget)

* **No trader-profile inputs:** block B removed from the observation.
* **No candidate-policy performance features:** the shadow-performance part of block D removed (current proposals and confidences remain).

Each ablation is trained and early-stopped on the same validation split as the full model. They are compared on the test split only at the end.

In [ ]:
# ---- 10a. Train ablation selectors ----
ABLATION_MODELS, ABLATION_ENVCFG, ABLATION_INFO = {}, {}, {}
if CFG.run_ablations:
    for name, (use_prof, use_perf) in {"PPO ablation: no trader profile": (False, True), "PPO ablation: no candidate-perf features": (True, False)}.items():
        ec = copy.deepcopy(CFG.env); ec.use_trader_profile, ec.use_candidate_perf = use_prof, use_perf
        m, h, info = train_selector("abl_" + ("noprofile" if not use_prof else "noperf"), ec, CFG.reward, CFG.risk, CFG.ppo)
        ABLATION_MODELS[name], ABLATION_ENVCFG[name], ABLATION_INFO[name] = m, ec, info
else:
    print("Ablations skipped (CFG.run_ablations=False).")

## 9. Held-out backtest and 10b. Baselines

All strategies run through **the same environment code**: identical test dates, initial capital, fees/slippage, risk gate and investor profiles. Baselines are the five specialists alone, a regime-based heuristic selector, a random selector (sanity reference), a gated buy-and-hold, an SMA-crossover rule, and an *ungated* 100%-invested buy-and-hold as a market reference.

In [ ]:
# ---- 9/10b. Held-out test backtest: PPO + ablations + baselines ----
class FixedSelector:
    def __init__(self, k): self.k = k
    def __call__(self, obs, env=None): return self.k
REGIME_TO_POLICY = {"up_lowvol": 2, "up_highvol": 0, "sideways_lowvol": 1, "sideways_highvol": 3, "down_lowvol": 4, "down_highvol": 4}
class RegimeHeuristicSelector:
    """Rule-based selector using only the past-only regime label (no learning)."""
    def __call__(self, obs, env): return REGIME_TO_POLICY.get(env.current_regime(), 4)
class RandomSelector:
    def __init__(self, seed): self.rng = np.random.default_rng(seed)
    def __call__(self, obs, env=None): return int(self.rng.integers(N_POLICIES))

_SMA50_IDX = MARKET_FEATURES.index("close_sma50")
def rule_buy_and_hold_gated(env): return BUY                                   # keep buying until the gate's position cap binds
def rule_sma_cross(env): return BUY if env.b.market_raw[env.t, _SMA50_IDX] > 0 else EXIT   # close > 50-day SMA -> long, else flat

def buy_and_hold_ungated(store: DataStore, ticker: str, profile: InvestorProfile, ec: EnvConfig) -> dict:
    """Market reference: 100% invested at the first test open, same fee/slippage, no risk gate."""
    b = store.bundles[ticker]; i0, i1 = store.split_idx["test"][ticker]
    fee, slip = ec.fee_bps / 1e4, ec.slippage_bps / 1e4; px = b.open[i0 + 1] * (1 + slip)
    shares = ec.initial_capital / (1 + fee) / px; cash = ec.initial_capital - shares * px * (1 + fee)
    eq = [ec.initial_capital] + [cash + shares * b.close[t + 1] for t in range(i0, i1)]
    rets = np.diff(eq) / np.array(eq[:-1])
    return dict(ticker=ticker, profile=profile, equity=np.array(eq), dates=b.dates[i0:i1 + 1], rets=rets, regimes=b.regime[i0:i1],
                steps=None, risk_log=pd.DataFrame(), traded=np.array([shares * px]), rejected=0, overrides=0, trades=1, decisions=0,
                exec_counts=np.zeros(4), behavior={}, reward_parts={}, total_reward=np.nan)

def evaluate_on_test(name, kind, obj=None, env_cfg=None) -> List[dict]:
    ec = env_cfg or CFG.env; out = []
    if kind == "ungated_bh":
        return [buy_and_hold_ungated(STORE, t, p, ec) for t in STORE.tickers for p in PROFILES_TEST]
    env = make_env("test", PROFILES_TEST, ec, CFG.reward, CFG.risk, "sequential", record=True)
    for _ in range(len(env.combos)):
        out.append(run_episode_selector(env, obj) if kind == "selector" else run_episode_rule(env, obj))
    return out

TEST_RUNS: Dict[str, List[dict]] = {}
TEST_RUNS["PPO Meta-RL (full)"] = evaluate_on_test("PPO Meta-RL (full)", "selector", PPOSelector(MODEL_FULL))
for k in range(N_POLICIES): TEST_RUNS[f"{POLICY_NAMES[k]} alone"] = evaluate_on_test("", "selector", FixedSelector(k))
TEST_RUNS["Regime-heuristic selector"] = evaluate_on_test("", "selector", RegimeHeuristicSelector())
TEST_RUNS["Random selector (reference)"] = evaluate_on_test("", "selector", RandomSelector(CFG.seed))
TEST_RUNS["Buy-and-hold (gated, same cap)"] = evaluate_on_test("", "rule", rule_buy_and_hold_gated)
TEST_RUNS["SMA-50 rule (gated)"] = evaluate_on_test("", "rule", rule_sma_cross)
TEST_RUNS["Buy-and-hold (100%, ungated)"] = evaluate_on_test("", "ungated_bh")
for nm, m in ABLATION_MODELS.items(): TEST_RUNS[nm] = evaluate_on_test(nm, "selector", PPOSelector(m), ABLATION_ENVCFG[nm])
print(f"Evaluated {len(TEST_RUNS)} strategies x {len(TEST_RUNS['PPO Meta-RL (full)'])} (ticker x profile) runs on the held-out test split.")
print("Test windows:", {t: STORE.split_dates[t]["test"] for t in list(STORE.tickers)[:3]}, "...")

## 10c. Comparison tables, regime / investor-profile breakdowns, and an evidence-based verdict

Behavioural consistency (agreement of executed-action frequencies and holding durations with the profile's historical behaviour) and risk compliance (`investor_dd_breach`) are reported **separately** from financial performance. Synthetic profiles have no real history, so consistency here is a diagnostic of *compatibility with the profile's stated habits*, not evidence of profitability.

In [ ]:
# ---- 10c. Aggregate comparison ----
RUN_KEYS = [(s["ticker"], s["profile"].profile_id) for s in TEST_RUNS["PPO Meta-RL (full)"]]
METRIC_COLS = ["cum_return", "ann_return", "sharpe", "sortino", "max_drawdown", "turnover_ann", "trade_count", "rejection_rate",
               "rejection_rate_nonhold", "avg_exposure", "behavior_consistency", "investor_dd_breach"]
def per_run_metrics(name):
    rows = []
    for s in TEST_RUNS[name]:
        m = metrics_from_summary(s); m.update(ticker=s["ticker"], profile_id=s["profile"].profile_id, archetype=s["profile"].archetype); rows.append(m)
    return pd.DataFrame(rows)
PER_RUN = {n: per_run_metrics(n) for n in TEST_RUNS}

def pooled_equal_weight(name):
    """Equal-weight portfolio of all (ticker x profile) runs, aligned by date."""
    cols = {}
    for s in TEST_RUNS[name]:
        cols[(s["ticker"], s["profile"].profile_id)] = pd.Series(s["equity"] / s["equity"][0], index=pd.to_datetime(s["dates"]))
    return pd.DataFrame(cols).ffill().dropna().mean(axis=1)
POOLED = {n: pooled_equal_weight(n) for n in TEST_RUNS}

rows = []
for n, df in PER_RUN.items():
    r = {"strategy": n}; r.update(df[METRIC_COLS].mean(numeric_only=True).to_dict())
    pe = perf_from_equity(POOLED[n].values); r.update({f"pooled_{k}": pe[k] for k in ("cum_return", "sharpe", "max_drawdown")}); rows.append(r)
COMPARE = pd.DataFrame(rows).set_index("strategy")
print("HELD-OUT TEST: mean over ticker x investor-profile runs (pooled_* = equal-weight portfolio of all runs)")
display(COMPARE.round(4))

# Policy-selection frequency (PPO and ablations)
SEL_FREQ = {}
for n in ["PPO Meta-RL (full)"] + list(ABLATION_MODELS):
    st = pd.concat([s["steps"] for s in TEST_RUNS[n]]); SEL_FREQ[n] = st["selected_policy_id"].map(dict(enumerate(POLICY_NAMES))).value_counts(normalize=True).reindex(POLICY_NAMES).fillna(0)
SEL_FREQ = pd.DataFrame(SEL_FREQ); print("\nPolicy-selection frequency on the test split:"); display(SEL_FREQ.round(3))

# Performance by market regime
REGIME_DF = pd.concat([regime_table(TEST_RUNS[n], n) for n in TEST_RUNS if n != "Buy-and-hold (100%, ungated)"] + [regime_table(TEST_RUNS["Buy-and-hold (100%, ungated)"], "Buy-and-hold (100%, ungated)")])
print("\nPerformance by market regime (regime label is past-only; return is the next-day realised portfolio return):")
display(REGIME_DF.pivot(index="regime", columns="strategy", values="ann_sharpe").round(2))
print("Days per regime:", REGIME_DF[REGIME_DF.strategy == "PPO Meta-RL (full)"].set_index("regime")["n_days"].to_dict())

# Performance by investor risk profile (synthetic archetypes)
ARCH_DF = pd.concat([df.assign(strategy=n) for n, df in PER_RUN.items()]).groupby(["archetype", "strategy"])[["cum_return", "sharpe", "max_drawdown", "rejection_rate", "behavior_consistency", "investor_dd_breach"]].mean()
print("\nPerformance by investor risk profile (SYNTHETIC archetypes) – showing PPO and the best fixed specialist:")
display(ARCH_DF.loc[(slice(None), ["PPO Meta-RL (full)", "Regime-heuristic selector", "Buy-and-hold (gated, same cap)"]), :].round(4))

# Evidence-based verdict (paired bootstrap over runs; runs share dates so the CI is optimistic)
def paired_bootstrap(a: pd.DataFrame, b: pd.DataFrame, col="sharpe", n=2000, seed=0):
    d = (a.set_index(["ticker", "profile_id"])[col] - b.set_index(["ticker", "profile_id"])[col]).dropna().values
    rng = np.random.default_rng(seed); bs = [rng.choice(d, len(d)).mean() for _ in range(n)]
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
ppo = PER_RUN["PPO Meta-RL (full)"]; BOOT = {}
for n in PER_RUN:
    if n != "PPO Meta-RL (full)": BOOT[n] = paired_bootstrap(ppo, PER_RUN[n])
BOOT_DF = pd.DataFrame(BOOT, index=["mean_sharpe_diff", "ci95_low", "ci95_high"]).T
print("\nPaired bootstrap of (PPO full – other) Sharpe difference across runs:"); display(BOOT_DF.round(3))
better_all = (BOOT_DF["ci95_low"] > 0).all(); better_mean = (BOOT_DF["mean_sharpe_diff"] > 0).all()
if better_all: VERDICT = "On this held-out window PPO's mean Sharpe exceeds every comparator with a bootstrap CI above zero (CIs are optimistic: runs share dates and one training seed was used)."
elif better_mean: VERDICT = "PPO's mean Sharpe is higher than every comparator, but at least one paired CI includes zero: the evidence is NOT sufficient to claim superiority."
else:
    lose = BOOT_DF.index[BOOT_DF["mean_sharpe_diff"] <= 0].tolist(); VERDICT = f"The held-out results do NOT support a claim that Meta-RL is superior: it does not beat {lose}."
print("\nVERDICT:", VERDICT)
if "PPO ablation: no trader profile" in BOOT_DF.index:
    for n in ("PPO ablation: no trader profile", "PPO ablation: no candidate-perf features"):
        d, lo, hi = BOOT_DF.loc[n, ["mean_sharpe_diff", "ci95_low", "ci95_high"]]
        print(f"Ablation check – full minus '{n}': {d:+.3f} Sharpe (95% CI {lo:+.3f}..{hi:+.3f}) -> " + ("component adds measurable value here." if lo > 0 else "no reliable evidence that this component helps on this test window."))

## 11. Visualisation

In [ ]:
# ---- 11. Plots ----
fig, ax = plt.subplots(2, 2, figsize=(16, 9))
hl = ["PPO Meta-RL (full)", "Regime-heuristic selector", "Buy-and-hold (gated, same cap)", "Buy-and-hold (100%, ungated)", "PPO ablation: no trader profile", "PPO ablation: no candidate-perf features"]
for n in TEST_RUNS:
    lw = 2.6 if n == "PPO Meta-RL (full)" else 1.0; alpha = 1 if n in hl else .45
    ax[0, 0].plot(POOLED[n].index, POOLED[n].values, lw=lw, alpha=alpha, label=n)
ax[0, 0].set_title("Equity (equal-weight over runs, start = 1.0), held-out test"); ax[0, 0].legend(fontsize=6, ncol=2); ax[0, 0].grid(alpha=.3)
for n in hl:
    if n in POOLED: ax[0, 1].plot(POOLED[n].index, POOLED[n].values / POOLED[n].cummax() - 1, label=n)
ax[0, 1].set_title("Drawdown (pooled)"); ax[0, 1].grid(alpha=.3); ax[0, 1].legend(fontsize=6)
SEL_FREQ.plot(kind="bar", ax=ax[1, 0]); ax[1, 0].set_title("Policy-selection frequency (test)"); ax[1, 0].tick_params(axis="x", rotation=25); ax[1, 0].legend(fontsize=6)
c = COMPARE[["sharpe", "behavior_consistency"]].sort_values("sharpe"); ax[1, 1].barh(c.index, c["sharpe"]); ax[1, 1].set_title("Mean Sharpe by strategy (test)"); ax[1, 1].tick_params(axis="y", labelsize=7); ax[1, 1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# Selection over time + regime context for one run
st = TEST_RUNS["PPO Meta-RL (full)"][0]["steps"]; fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(pd.to_datetime(st.date), st.equity_after, c="k"); ax[0].set_title(f"PPO equity – {st.ticker.iloc[0]} / {st.profile_id.iloc[0]} (synthetic profile)")
ax[1].scatter(pd.to_datetime(st.date), st.selected_policy_id, c=st.selected_policy_id, cmap="tab10", s=8); ax[1].set_yticks(range(N_POLICIES)); ax[1].set_yticklabels(POLICY_NAMES, fontsize=7)
plt.tight_layout(); plt.show()

# Regime heat-map and risk-gate rejection reasons
pv = REGIME_DF.pivot(index="regime", columns="strategy", values="ann_sharpe")[["PPO Meta-RL (full)", "Regime-heuristic selector", "Buy-and-hold (gated, same cap)", "Buy-and-hold (100%, ungated)"]]
fig, ax = plt.subplots(1, 2, figsize=(15, 4)); im = ax[0].imshow(pv.values, cmap="RdYlGn", aspect="auto"); ax[0].set_xticks(range(pv.shape[1])); ax[0].set_xticklabels(pv.columns, rotation=20, fontsize=7)
ax[0].set_yticks(range(pv.shape[0])); ax[0].set_yticklabels(pv.index); ax[0].set_title("Annualised Sharpe by regime"); plt.colorbar(im, ax=ax[0])
rl = pd.concat([s["risk_log"] for s in TEST_RUNS["PPO Meta-RL (full)"] if len(s["risk_log"])]) if any(len(s["risk_log"]) for s in TEST_RUNS["PPO Meta-RL (full)"]) else pd.DataFrame({"code": []})
if len(rl): rl["code"].value_counts().plot(kind="bar", ax=ax[1], color="C3")
ax[1].set_title("Risk-gate rejections / clips by reason (PPO, test)"); plt.tight_layout(); plt.show()

## 12. Model export

Writes `meta_rl_selector.zip` (SB3 PPO checkpoint), `meta_rl_config.json`, `meta_rl_observation_schema.json`, `risk_controller_config.json`, `evaluation_metrics.json`, `policy_selection_log.csv`, `equity_curve.csv`, a rejection log and an integration report. Artifact SHA-256 hashes are stored so inference can verify that it is paired with the *same* five specialists used in training.

In [ ]:
# ---- 12. Export ----
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return _clean(o.tolist())
    return o
def write_json(obj, name): (OUT / name).write_text(json.dumps(_clean(obj), indent=2)); return OUT / name

MODEL_FULL.save(OUT / "meta_rl_selector")                                         # -> meta_rl_selector.zip
LAYOUT = observation_names(CFG.env.use_trader_profile, CFG.env.use_candidate_perf)
nz = STORE.norm
schema = dict(
    schema_version="1.0", dim=len(LAYOUT), dtype="float32", clip=[-10, 10],
    block_order=["market", "trader_behavior", "investor_position", "candidate_policies"],
    use_trader_profile=CFG.env.use_trader_profile, use_candidate_perf=CFG.env.use_candidate_perf,
    canonical_action_order=ACTIONS, policy_order=POLICY_NAMES, selector_action_space={str(i): n for i, n in enumerate(SELECT_NAMES)},
    features=[dict(index=i, block=b, name=n) for i, (b, n) in enumerate(LAYOUT)],
    normalization=dict(fit_on="train split only", **nz.to_dict(), market_feature_order=MARKET_FEATURES, flag_features_not_scaled=FLAG_FEATURES, shadow_metric_order=SHADOW_METRICS),
    behavior_block=dict(features=BEHAVIOR_FEATURES, source="synthetic profiles unless a real trade history was supplied", encoder_embeddings_used=False),
    position_block=dict(features=POSITION_FEATURES, checkpoint_days=CFG.env.checkpoint_days),
    policy_feature_columns=POLICY_COLUMNS, policy_seq_len=SEQ_LEN)
write_json(schema, "meta_rl_observation_schema.json")
write_json(asdict(CFG.risk), "risk_controller_config.json")
config = dict(
    model_name="Personalized Meta-RL Strategy Selector", algorithm="PPO (stable-baselines3)", created_utc=pd.Timestamp.utcnow().isoformat(),
    versions=dict(python=platform.python_version(), torch=torch.__version__, stable_baselines3=sb3.__version__, gymnasium=gym.__version__, numpy=np.__version__, pandas=pd.__version__),
    seed=CFG.seed, config=dict(data=asdict(CFG.data), env=asdict(CFG.env), reward=asdict(CFG.reward), risk=asdict(CFG.risk), profiles=asdict(CFG.profiles), ppo=asdict(CFG.ppo)),
    training=INFO_FULL, ablation_training=ABLATION_INFO, split_dates=STORE.split_dates,
    artifacts=dict(artifact_dir=ARTIFACT_META["artifact_dir"], files=ARTIFACT_META["files"], policy_names=POLICY_NAMES, action_map=POLICIES[0].action_map),
    investor_profiles=dict(source=sorted({p.source for p in PROFILES_TRAIN + PROFILES_VAL + PROFILES_TEST}),
                           note="Synthetic profiles are experimental proxies; no claim is made about any real individual's behaviour."),
    mode_restriction="simulation/paper only; live trading is not authorised by the risk controller",
    disclaimer="Research prototype for offline evaluation and paper trading. Simulated results do not guarantee live profitability.")
write_json(config, "meta_rl_config.json")

agg = {n: {k: (None if not np.isfinite(v) else float(v)) for k, v in COMPARE.loc[n].items()} for n in COMPARE.index}
evaluation = dict(split="held-out test (never used for selection)", test_windows={t: STORE.split_dates[t]["test"] for t in STORE.tickers},
                  n_runs=len(RUN_KEYS), initial_capital=CFG.env.initial_capital, fee_bps=CFG.env.fee_bps, slippage_bps=CFG.env.slippage_bps,
                  strategies=agg, policy_selection_frequency=SEL_FREQ.to_dict(), by_regime=REGIME_DF.to_dict(orient="records"),
                  by_investor_archetype=ARCH_DF.reset_index().to_dict(orient="records"), paired_bootstrap_sharpe=BOOT_DF.to_dict(orient="index"),
                  verdict=VERDICT, reward_sensitivity_validation=(SENS_DF.to_dict(orient="index") if len(SENS_DF) else {}),
                  validation_history=HIST_FULL, caveats=["single training seed unless repeated", "synthetic investor profiles", "bootstrap CIs treat runs as independent although they share dates"])
write_json(evaluation, "evaluation_metrics.json")

sel_log = pd.concat([s["steps"] for s in TEST_RUNS["PPO Meta-RL (full)"]], ignore_index=True)
sel_log.to_csv(OUT / "policy_selection_log.csv", index=False)
pd.concat([s["risk_log"] for s in TEST_RUNS["PPO Meta-RL (full)"] if len(s["risk_log"])] or [pd.DataFrame()]).to_csv(OUT / "risk_rejection_log.csv", index=False)
eq_rows = []
for n, runs in TEST_RUNS.items():
    for s in runs: eq_rows.append(pd.DataFrame({"strategy": n, "ticker": s["ticker"], "profile_id": s["profile"].profile_id, "date": pd.to_datetime(s["dates"]).strftime("%Y-%m-%d"), "equity": s["equity"]}))
pd.concat(eq_rows).to_csv(OUT / "equity_curve.csv", index=False)
print("Saved:", sorted(p.name for p in OUT.iterdir() if p.is_file()))
if CFG.paths.drive_output_dir and Path("/content/drive").exists():
    import shutil; Path(CFG.paths.drive_output_dir).mkdir(parents=True, exist_ok=True)
    for p in OUT.iterdir():
        if p.is_file(): shutil.copy(p, CFG.paths.drive_output_dir)
    print("Copied outputs to", CFG.paths.drive_output_dir)
try:
    from google.colab import files as _cf   # type: ignore
    for nm in ["meta_rl_selector.zip", "meta_rl_config.json", "meta_rl_observation_schema.json", "risk_controller_config.json", "evaluation_metrics.json", "policy_selection_log.csv", "equity_curve.csv"]: _cf.download(str(OUT / nm))
except Exception: pass

## 13. Inference interface

`load_meta_agent(path)`, `select_policy(observation)`, `predict_action(observation)` and `save_selection_log(path)` are thin wrappers over a `MetaAgent`. Four concerns stay in **separate functions**: (1) model selection (`select_policy`), (2) policy inference (`policy_proposal` / `build_observation`), (3) risk validation (`RiskController.validate`), (4) simulated execution (`execute_order`).

`predict_action` takes the chosen specialist's *actual* proposal and confidence from the observation and always passes it through the risk controller. It returns the final action, the risk decision and the rejection reason where applicable.

In [ ]:
# ---- 13. Inference interface ----
@dataclass
class Observation:
    vector: np.ndarray            # normalised observation (schema in meta_rl_observation_schema.json)
    policy_probs: np.ndarray      # (5,4) action probabilities of each frozen specialist, canonical order HOLD,BUY,SELL,EXIT
    policy_conf: np.ndarray       # (5,) confidence (max probability, uncalibrated)
    risk_state: RiskState
    profile: InvestorProfile
    ticker: str = ""
    timestamp: str = ""

class MetaAgent:
    def __init__(self, model: PPO, config: dict, schema: dict, policies: List[FrozenPolicy], risk_cfg: RiskConfig):
        self.model, self.config, self.schema, self.policies, self.risk_cfg = model, config, schema, policies, risk_cfg
        self.norm = Normalizer.from_dict(schema["normalization"]); self.risk = RiskController(risk_cfg)
        self.use_profile, self.use_perf = schema["use_trader_profile"], schema["use_candidate_perf"]
        self.env_cfg = EnvConfig(**config["config"]["env"]); self.selector = PPOSelector(model); self.selection_log: List[dict] = []
        if model.observation_space.shape[0] != schema["dim"]: raise ValueError("Model observation dim does not match the schema.")

    # (2) policy inference + observation construction (training/inference share the same code path)
    def build_observation(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame, profile: InvestorProfile, portfolio_state: Optional[dict] = None, ticker: str = "LIVE") -> Observation:
        df, bench = _clean_ohlcv(ohlcv, ticker), _clean_ohlcv(benchmark, "benchmark")
        b = build_bundle(ticker, df, bench, self.policies, self.env_cfg, self.risk_cfg); t = len(df) - 1
        if t < b.valid_from: raise ValueError(f"Not enough history: need > {b.valid_from} rows of OHLCV (use the full history from the same start date as training; shadow metrics are path-dependent).")
        ps = dict(position_frac=0.0, cash_frac=1.0, unrealized_pnl=0.0, hold_days=0, drawdown=0.0, last_day_return=0.0, steps_in_period=0,
                  equity=self.env_cfg.initial_capital, peak_equity=self.env_cfg.initial_capital, mode="simulation"); ps.update(portfolio_state or {})
        pos = position_vector(ps["position_frac"], ps["unrealized_pnl"], ps["hold_days"], ps["drawdown"], ps["last_day_return"], ps["steps_in_period"], profile, self.env_cfg.checkpoint_days)
        vec = assemble_observation(self.norm.market(b.market_raw[t]), profile.behavior_vector(), pos, np.concatenate([b.probs[t].reshape(-1), b.conf[t]]),
                                   self.norm.shadow(b.shadow_raw[t].reshape(-1)), self.use_profile, self.use_perf)
        rs = RiskState(ps["position_frac"], ps["cash_frac"], ps["equity"], ps["peak_equity"], ps["drawdown"], ps["last_day_return"], float(b.ann_vol[t]), ps["mode"])
        return Observation(vec, b.probs[t].copy(), b.conf[t].copy(), rs, profile, ticker, str(df.index[-1].date()))

    # (1) model selection
    def select_policy(self, observation) -> dict:
        vec = observation.vector if isinstance(observation, Observation) else np.asarray(observation, np.float32)
        if vec.shape[-1] != self.schema["dim"]: raise ValueError(f"Observation has dim {vec.shape[-1]}, expected {self.schema['dim']}.")
        p = self.selector.probs(vec); k = int(np.argmax(p))
        return dict(policy_id=k, policy_name=POLICY_NAMES[k], selection_probabilities={POLICY_NAMES[i]: float(p[i]) for i in range(N_POLICIES)})

    # (2) the chosen specialist's proposal
    def policy_proposal(self, observation: Observation, policy_id: int) -> dict:
        a = int(np.argmax(observation.policy_probs[policy_id]))
        return dict(action=ACTIONS[a], action_id=a, confidence=float(observation.policy_conf[policy_id]), probabilities=dict(zip(ACTIONS, observation.policy_probs[policy_id].astype(float))))

    # (3) risk validation, always applied
    def predict_action(self, observation) -> dict:
        if not isinstance(observation, Observation):
            raise ValueError("predict_action needs an Observation (from build_observation) so the proposal and risk state are available.")
        sel = self.select_policy(observation); prop = self.policy_proposal(observation, sel["policy_id"])
        dec = self.risk.validate(prop["action_id"], observation.risk_state, observation.profile)
        out = dict(final_action=ACTIONS[dec.final_action], proposed_action=prop["action"], proposal_confidence=prop["confidence"],
                   selected_policy_id=sel["policy_id"], selected_policy_name=sel["policy_name"], selection_probabilities=sel["selection_probabilities"],
                   risk_decision=("override" if dec.override else "approved" if dec.approved else "rejected"), risk_reason_code=dec.reason_code,
                   rejection_reason=(None if dec.approved else dec.reason), order_fraction=dec.order_fraction, clipped=dec.clipped, mode=observation.risk_state.mode)
        self.selection_log.append(dict(timestamp=observation.timestamp, ticker=observation.ticker, profile_id=observation.profile.profile_id, **{k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in out.items()}))
        return out

    def save_selection_log(self, path):
        pd.DataFrame(self.selection_log).to_csv(path, index=False); return str(path)

_ACTIVE_AGENT: Optional[MetaAgent] = None
def load_meta_agent(path, artifact_dir=None, verify_hashes: bool = True) -> MetaAgent:
    global _ACTIVE_AGENT
    path = Path(path); d = path.parent
    config = json.loads((d / "meta_rl_config.json").read_text()); schema = json.loads((d / "meta_rl_observation_schema.json").read_text())
    rd = json.loads((d / "risk_controller_config.json").read_text()); rd["allowed_modes"] = tuple(rd["allowed_modes"])
    model = PPO.load(str(path), device="cpu")
    policies, _, meta = load_all_policies(Path(artifact_dir or config["artifacts"]["artifact_dir"]))
    if verify_hashes and meta["files"] != config["artifacts"]["files"]:
        raise ArtifactError("Specialist-policy artifacts differ (SHA-256) from those used to train this selector. Refusing to pair them.")
    _ACTIVE_AGENT = MetaAgent(model, config, schema, policies, RiskConfig(**rd)); return _ACTIVE_AGENT
def select_policy(observation): return _ACTIVE_AGENT.select_policy(observation)
def predict_action(observation): return _ACTIVE_AGENT.predict_action(observation)
def save_selection_log(path): return _ACTIVE_AGENT.save_selection_log(path)

# ---- Demonstration (paper/simulation mode) ----
agent = load_meta_agent(OUT / "meta_rl_selector.zip")
tk, prof = STORE.tickers[0], PROFILES_TEST[0]; df_demo = load_ohlcv(tk)
obs = agent.build_observation(df_demo, BENCH_DF, prof, dict(position_frac=0.10, cash_frac=0.90, unrealized_pnl=0.02, hold_days=7, drawdown=0.01), ticker=tk)
print("select_policy ->", json.dumps(select_policy(obs), indent=1))
print("predict_action ->", json.dumps(predict_action(obs), indent=1, default=str))
# Train/inference parity: the live-style builder must reproduce the environment's observation for the same date and state.
t_last = len(df_demo) - 1; A = STORE.arrays(tk)
ref = assemble_observation(A["market"][t_last], prof.behavior_vector(), position_vector(0.10, 0.02, 7, 0.01, 0.0, 0, prof, CFG.env.checkpoint_days),
                           A["core"][t_last], A["perf"][t_last], CFG.env.use_trader_profile, CFG.env.use_candidate_perf)
assert np.allclose(ref, obs.vector, atol=1e-4), "Train/inference observation mismatch!"; print("Train/inference observation parity check passed.")
# The gate rejects/overrides regardless of PPO's choice:
import dataclasses as _dc
for label, rs in [("drawdown breach", _dc.replace(obs.risk_state, drawdown=0.95)), ("live mode requested", _dc.replace(obs.risk_state, mode="live")), ("kill-switch off", obs.risk_state)]:
    o2 = _dc.replace(obs, risk_state=rs)
    if label == "kill-switch off": agent.risk.cfg = _dc.replace(agent.risk.cfg, trading_enabled=False)
    r = predict_action(o2); print(f"{label:>20}: proposed {r['proposed_action']} -> final {r['final_action']} | {r['risk_decision']} | {r['risk_reason_code']}")
agent.risk.cfg = _dc.replace(agent.risk.cfg, trading_enabled=True)
print("Saved selection log:", save_selection_log(OUT / "inference_demo_selection_log.csv"))

In [ ]:
# ---- Final integration report ----
from IPython.display import Markdown, display as _display
art_rows = "\n".join(f"| {POLICY_NAMES[i]} | `{ARTIFACT_FILES[i]['model']}` | `{ARTIFACT_FILES[i]['config']}` | `{ARTIFACT_FILES[i]['schema']}` | `{ARTIFACT_FILES[i]['preprocessor']}` | `{ARTIFACT_META['files'][ARTIFACT_FILES[i]['model'][:-3]]['model']['sha256'][:12]}…` |" for i in range(N_POLICIES))
REPORT = f"""# Integration report – Personalized Meta-RL Strategy Selector

**Status:** research prototype for offline evaluation and paper trading. Simulated results do not guarantee live profitability, and live trading is not authorised by the risk controller (`allowed_modes = {list(CFG.risk.allowed_modes)}`).

**Held-out verdict (computed, not assumed):** {VERDICT}

## 1. Required input artifacts (all validated at load time)
| Policy | Model | Config | Feature schema | Preprocessor | Model SHA-256 |
|---|---|---|---|---|---|
{art_rows}

Shared contract verified: {len(POLICY_COLUMNS)} feature columns in identical order, `seq_len={SEQ_LEN}`, `action_map={POLICIES[0].action_map}`, forward-pass shape check. Config keys used: `action_map`, `output_type`, `input_mode`, `preprocessing`, `train_end_date`, optional `arch`. Schema keys: `feature_columns`, `seq_len`.

## 2. Observation schema (`meta_rl_observation_schema.json`)
{len(LAYOUT)} float32 dims = market ({len(MARKET_FEATURES)}) + trader behaviour ({len(BEHAVIOR_FEATURES) if CFG.env.use_trader_profile else 0}) + investor/position ({len(POSITION_FEATURES)}) + candidate policies ({len(candidate_names(CFG.env.use_candidate_perf))}). Market/shadow features are z-scored with train-split statistics (stored in the schema), clipped to ±5; the whole vector is clipped to ±10.

## 3. Output interface
| Function | Returns |
|---|---|
| `load_meta_agent(path)` | `MetaAgent` (verifies specialist SHA-256 hashes) |
| `select_policy(observation)` | `policy_id`, `policy_name`, `selection_probabilities` |
| `predict_action(observation)` | `final_action`, `proposed_action`, `proposal_confidence`, selected policy, `risk_decision` (approved/rejected/override), `risk_reason_code`, `rejection_reason`, `order_fraction`, `mode` |
| `save_selection_log(path)` | CSV of every `predict_action` call |

Exported files: `meta_rl_selector.zip`, `meta_rl_config.json`, `meta_rl_observation_schema.json`, `risk_controller_config.json`, `evaluation_metrics.json`, `policy_selection_log.csv`, `equity_curve.csv`, `risk_rejection_log.csv`.

## 4. Connecting to a future FastAPI backend
1. **Package the models:** put `meta_rl_selector.zip`, the three JSON files and the five specialist artifact sets (`.pt`, config, schema, preprocessor) in one versioned `models/` directory; keep `meta_rl_config.json -> artifacts.files` hashes.
2. **Extract code into a module** (e.g. `meta_rl/`): constants, `FrozenPolicy` + loaders, feature builders (`build_market_frame`, `policy_probabilities`, `shadow_features`, `build_bundle`), `InvestorProfile`, `Normalizer`, `position_vector`, `assemble_observation`, `RiskController`, `execute_order` (simulation only), `PPOSelector`, `MetaAgent`. Replace the notebook feature builder with the exact feature code from notebooks 1-5.
3. **Load once** at startup (FastAPI lifespan): `agent = load_meta_agent("models/meta_rl_selector.zip")`; fail fast on hash mismatch.
4. **Endpoints:** `POST /v1/meta/select` (history + profile + portfolio state → `select_policy`) and `POST /v1/meta/act` (→ `predict_action`). Request: ticker, full OHLCV history from the *same start date used in training* (shadow metrics are path-dependent), benchmark history, investor profile (declared limits and behaviour statistics), portfolio state (`position_frac`, `cash_frac`, `unrealized_pnl`, `hold_days`, `drawdown`, `equity`, `peak_equity`, `last_day_return`, `mode`). Validate with Pydantic; reject any `mode` other than `simulation`/`paper`.
5. **Keep the risk controller authoritative:** the API must never return PPO's selection as an order; only `predict_action`'s validated result. Persist every request/decision (audit trail) and expose the rejection reasons.
6. **Parity test in CI:** replay this notebook's test dates through the API and assert the observation vector equals the environment's (the notebook performs this check once).
7. **Operations:** monitor selection frequency, rejection rate and feature drift; define retraining triggers; retrain/re-validate the selector whenever any specialist policy changes (hashes will mismatch by design).
8. **Before any real-money use:** independent validation, multi-seed training, broker/exchange realism (lot sizes, borrow, halts), compliance review, and a human kill switch (`trading_enabled`).

## 5. Known limitations
* Synthetic investor profiles are experimental proxies; no real individual's behaviour was learned unless a real `trader_history_csv` was supplied (and then only as a test profile).
* Single training seed per model by default; PPO results vary across seeds.
* Execution model: long-only, fractional shares, fills at next open ± slippage, no market impact or liquidity limits.
* Paired-bootstrap CIs treat runs as independent although they share dates, so they are optimistic.
* Specialist policies were assumed to depend only on market features (proposals are not conditioned on portfolio state).
"""
(OUT / "integration_report.md").write_text(REPORT); _display(Markdown(REPORT))